<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">07 Demo - Shuffle, Skew, Spill and Explosion</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Shuffle, Skew, Spill and Explosion
#### Duration: 40 minutes
## Overview

Four distributed slow-downs cause most of the pain in a Spark workload, and they share a root cause: data moving through the network in ways that concentrate work unevenly. This part turns the measurement toolkit from Part 2 onto those four patterns (shuffle, skew, spill, and explosion).

You run a handful of deliberately shaped queries over the retail demo data, read each pattern in the **Query Profile** and its **Performance Insights**, then apply the cheapest fix the platform has not already made for you.

Each section follows the same approach. First, identify the pattern from the profile. Then check whether the platform's automatic handling (AQE, Photon) already absorbed it. Only intervene where a measurable gap remains.

> Serverless-first: this demo runs on a serverless notebook. Photon and AQE already absorb most of what engineers used to tune by hand, so your primary job here is recognizing the pattern in the profile and confirming which automatic optimizations engaged.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Distinguish** shuffle, skew, spill and explosion in a Query Profile and its Performance Insights.
2. **Explain** how Spark picks a join strategy and how AQE switches to a broadcast join at runtime.
3. **Apply** the skew fix order: let AQE handle skew first, filter null and junk keys next, and salt only as a last resort.
4. **Differentiate** a spill precursor (peak shuffle memory) from real spill, and name what causes each.
5. **Fix** a row explosion by pre-aggregating or using higher-order functions instead of exploding and re-collecting.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.query.history</code>, which lags live activity by <strong>5 to 10 minutes</strong>. Run the query cells first, then continue with the explanatory sections. By the time you reach the system-table cells, the rows will have landed. If a cell returns no rows, wait a few minutes and re-run it.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It builds the shared retail demo tables and stages three small helper tables for this part:

- **channel_dim**: a tiny 4 row lookup that the optimizer broadcasts (the small-side contrast to the 5M row dimensions).
- **order_items_dirty**: a copy of **order_items** where about 15% of rows carry a null **order_idx** (a broken upstream extract).
- **order_baskets**: line items rolled up into array columns, for the explosion and higher-order-function section.

It also defines the pattern query strings this notebook profiles (one set per section) and a `spill_shuffle_history()` helper.

In [ ]:
%run ./Includes/Classroom-Setup-07

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-label { font-size: 14pt; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">Learn to read the four patterns:</div>
    <div class="ds-banner-tagline">Each one has a name in Performance Insights and a signature in the profile.</div>
  </div>
</div>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-label">Shuffle</div>
    <div class="ds-card-title">Data crosses the network</div>
    <ul>
      <li>Wide transforms redistribute rows by key.</li>
      <li>Signature: Exchange operators, high shuffle bytes.</li>
      <li>The join strategy the optimizer picks determines the cost.</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="ds-card-label">Skew</div>
    <div class="ds-card-title">One task does all the work</div>
    <ul>
      <li>A few hot keys own most of the rows.</li>
      <li>Signature: one straggler task, the <strong>DATA_SKEW</strong> insight.</li>
      <li>Fix order: let AQE handle it, then clean keys, then salt as last resort.</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-label">Spill</div>
    <div class="ds-card-title">A task outgrows its memory</div>
    <ul>
      <li>Working set spills to local disk.</li>
      <li>Signature: the <strong>DATA_SPILL</strong> insight, spilled bytes.</li>
      <li>On serverless you see the precursor, not real spill.</li>
    </ul>
  </div>
  <div class="ds-card amber">
    <div class="ds-card-label">Explosion</div>
    <div class="ds-card-title">Rows out far exceed rows in</div>
    <ul>
      <li>Fan-out joins and explode multiply rows.</li>
      <li>Signature: the <strong>EXPLODING_JOIN</strong> insight.</li>
      <li>Reduce the row multiplication before it enters the shuffle.</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Key concepts referenced above

- **Wide transform**: any operation that requires rows to move across the network so that matching keys land on the same task. Joins, `GROUP BY`, `DISTINCT`, global `ORDER BY`, and window functions are all wide. Narrow transforms (`select`, `filter`, `withColumn`) touch only the rows already present on a task and never shuffle.
- **Straggler task**: one task that takes dramatically longer than its peers because it received a disproportionate share of the data. In the Query Profile it shows as a single long bar in the task-duration distribution. The Performance Insights label is **DATA_SKEW**.
- **AQE (Adaptive Query Execution)**: a Spark runtime feature that re-optimizes the physical plan between stages using actual data statistics rather than estimates. It coalesces small shuffle partitions, switches sort-merge joins to broadcast joins when the shuffle output is small enough, and splits skewed partitions so no single task is stuck with a hot key.
- **Salting**: a last-resort skew fix where you append a random suffix to the hot join key, spreading its rows across multiple tasks. You then join on the salted key and union the results. It works, but it doubles the shuffle and complicates the query, so you try AQE and key cleanup first.
- **Exchange operator**: the physical operator in the Query Profile that represents a shuffle. Each Exchange redistributes data by a hash of the join or group key. Color the profile by **Memory peak** to see which Exchanges carry the most data.
- **Peak shuffle memory**: the maximum memory a single task consumed during a shuffle. On serverless, Photon allocates enough memory to avoid spill at typical scale, so peak shuffle memory is the observable leading indicator of where spill would occur on constrained compute.
- **Spill**: when a task's working set exceeds its available memory and Spark writes intermediate data to local disk. Shows as `spilled_local_bytes > 0` in query history and the **DATA_SPILL** insight in Performance Insights.
- **Fan-out join / row explosion**: a join where rows on one side match many rows on the other, producing far more output rows than input rows. The Performance Insights label is **EXPLODING_JOIN**. Common causes include joining on a non-unique key or exploding array columns.
- **Higher-order functions**: SQL and PySpark functions (`TRANSFORM`, `FILTER`, `AGGREGATE`, `array_agg`) that operate on array elements in place, without exploding the array into separate rows. They avoid the row multiplication that `LATERAL VIEW EXPLODE` or `explode()` would cause.

### Why these four, and why together
- These are the four most common slow-down patterns and you will meet most often once a query is past the scan.
- They're related, not independent. For example skew concentrates rows onto one task, which is what pushes that task to spill. Explosion multiplies rows before a shuffle, which inflates both shuffle bytes and spill pressure.
- Each pattern has an exact Performance Insight name (**DATA_SKEW**, **DATA_SPILL**, **EXPLODING_JOIN**) and an operator-level signature in the Query Profile.

### The order of the sections
- Shuffle comes first because it's the mechanism underneath the other three. You cannot reason about skew or spill without understanding the redistribution that causes them.
- Skew, spill, and explosion then build on that shared vocabulary.
- The platform already handles a lot. Photon, AQE, and auto shuffle partitions absorb most of what used to be manual tuning. What remains is measuring which automatic optimizations engaged and intervening only where they left a measurable gap.

</details>

## B. Shuffle and Joins
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">A shuffle is the network cost of a wide transform</div>
    <div class="ds-banner-tagline">Joins, aggregations, distinct, and windows move rows across the network so matching keys land on the same task.</div>
  </div>
</div>


</div>



<div class="shufx">
  <style>
    .shufx { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .shufx h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .shufx p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .shufx svg { display: block; width: 100%; height: auto; }
    .shufx text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .shufx .ttl { font-size: 14px; font-weight: 650; fill: #14343f; }
    .shufx .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .shufx .sm { font-size: 12px; fill: #62707d; }
    .shufx .lbl { font-size: 13px; fill: #35424e; }
    .shufx .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .shufx .legend { display: flex; flex-wrap: wrap; gap: 8px 20px; margin-top: 14px; font-size: 12.5px; color: #4a5560; }
    .shufx .legend span.k { display: inline-flex; align-items: center; gap: 7px; }
    .shufx .sw { width: 13px; height: 13px; border-radius: 3px; display: inline-block; }
    .shufx .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .shufx code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>What Happens During a Shuffle</h3>
  <p class="sub">Rows move across the network so that matching keys land on the same task. This is the only cross-network data movement in Spark.</p>
  <svg viewBox="0 0 900 410" role="img" aria-label="Conceptual diagram: before a shuffle rows with mixed keys sit on random tasks, after the shuffle each task owns all rows for its assigned keys">
    <defs>
      <marker id="shf-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#c2703a"></path>
      </marker>
    </defs>
    <!-- ====== LEFT: BEFORE ====== -->
    <text class="hdr" x="16" y="20">BEFORE (scan output)</text>
    <text class="sm" x="16" y="36">Keys are scattered across tasks.</text>
    <!-- Task 0: y=50, h=52, center=76, bottom=102 -->
    <rect x="16" y="50" width="240" height="52" rx="8" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="tag" x="28" y="68" fill="#14343f">TASK 0</text>
    <rect x="28" y="74" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="38" y="88" fill="#0f6b7a">A</text>
    <rect x="78" y="74" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="88" y="88" fill="#7a4a2e">B</text>
    <rect x="128" y="74" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="138" y="88" fill="#2d6e3f">C</text>
    <rect x="178" y="74" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="188" y="88" fill="#0f6b7a">A</text>
    <!-- Task 1: y=118, h=52, center=144, bottom=170 -->
    <rect x="16" y="118" width="240" height="52" rx="8" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="tag" x="28" y="136" fill="#14343f">TASK 1</text>
    <rect x="28" y="142" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="38" y="156" fill="#2d6e3f">C</text>
    <rect x="78" y="142" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="88" y="156" fill="#7a4a2e">B</text>
    <rect x="128" y="142" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="138" y="156" fill="#0f6b7a">A</text>
    <rect x="178" y="142" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="188" y="156" fill="#7a4a2e">B</text>
    <!-- Task 2: y=186, h=52, center=212, bottom=238 -->
    <rect x="16" y="186" width="240" height="52" rx="8" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="tag" x="28" y="204" fill="#14343f">TASK 2</text>
    <rect x="28" y="210" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="38" y="224" fill="#7a4a2e">B</text>
    <rect x="78" y="210" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="88" y="224" fill="#0f6b7a">A</text>
    <rect x="128" y="210" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="138" y="224" fill="#2d6e3f">C</text>
    <rect x="178" y="210" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="188" y="224" fill="#2d6e3f">C</text>
    <!-- ====== MIDDLE: THE EXCHANGE ====== -->
    <rect x="280" y="32" width="340" height="248" rx="10" fill="#fdf3ec" stroke="#e5bfa2"></rect>
    <!-- Header: centered, well above arrows -->
    <text class="hdr" x="450" y="56" text-anchor="middle">EXCHANGE (shuffle)</text>
    <text class="sm" x="450" y="68" text-anchor="middle">hash(key) determines which task receives each row</text>
    <!-- arrows: task centers at y=76, y=144, y=212 -->
    <!-- A rows going to right task 0 -->
    <line x1="256" y1="76" x2="620" y2="76" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <line x1="256" y1="144" x2="620" y2="76" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <line x1="256" y1="212" x2="620" y2="76" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <!-- B rows going to right task 1 -->
    <line x1="256" y1="76" x2="620" y2="144" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <line x1="256" y1="144" x2="620" y2="144" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <line x1="256" y1="212" x2="620" y2="144" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <!-- C rows going to right task 2 -->
    <line x1="256" y1="76" x2="620" y2="212" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <line x1="256" y1="144" x2="620" y2="212" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <line x1="256" y1="212" x2="620" y2="212" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#shf-arr)"></line>
    <!-- Cost label: centered, below arrows -->
    <text class="tag" x="450" y="250" text-anchor="middle" fill="#7a4a2e">THIS IS THE NETWORK COST</text>
    <text class="sm" x="450" y="266" text-anchor="middle">shuffle_read_bytes in system_vw.query.history</text>
    <!-- ====== RIGHT: AFTER ====== -->
    <text class="hdr" x="640" y="20">AFTER (keys co-located)</text>
    <text class="sm" x="640" y="36">Each task owns ALL rows for its key.</text>
    <!-- Task 0: y=50, h=52, center=76 -->
    <rect x="640" y="50" width="240" height="52" rx="8" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="tag" x="652" y="68" fill="#14343f">TASK 0</text>
    <text class="sm" x="730" y="68" fill="#62707d">all key=A</text>
    <rect x="652" y="74" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="662" y="88" fill="#0f6b7a">A</text>
    <rect x="702" y="74" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="712" y="88" fill="#0f6b7a">A</text>
    <rect x="752" y="74" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="762" y="88" fill="#0f6b7a">A</text>
    <rect x="802" y="74" width="44" height="20" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="812" y="88" fill="#0f6b7a">A</text>
    <!-- Task 1: y=118, h=52, center=144 -->
    <rect x="640" y="118" width="240" height="52" rx="8" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="tag" x="652" y="136" fill="#14343f">TASK 1</text>
    <text class="sm" x="730" y="136" fill="#62707d">all key=B</text>
    <rect x="652" y="142" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="662" y="156" fill="#7a4a2e">B</text>
    <rect x="702" y="142" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="712" y="156" fill="#7a4a2e">B</text>
    <rect x="752" y="142" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="762" y="156" fill="#7a4a2e">B</text>
    <rect x="802" y="142" width="44" height="20" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="812" y="156" fill="#7a4a2e">B</text>
    <!-- Task 2: y=186, h=52, center=212 -->
    <rect x="640" y="186" width="240" height="52" rx="8" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="tag" x="652" y="204" fill="#14343f">TASK 2</text>
    <text class="sm" x="730" y="204" fill="#62707d">all key=C</text>
    <rect x="652" y="210" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="662" y="224" fill="#2d6e3f">C</text>
    <rect x="702" y="210" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="712" y="224" fill="#2d6e3f">C</text>
    <rect x="752" y="210" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="762" y="224" fill="#2d6e3f">C</text>
    <rect x="802" y="210" width="44" height="20" rx="4" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="812" y="224" fill="#2d6e3f">C</text>
    <!-- ====== BOTTOM: punchline ====== -->
    <rect x="16" y="290" width="864" height="108" rx="8" fill="#14343f"></rect>
    <text x="36" y="318" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Every wide transform (JOIN, GROUP BY, DISTINCT, ORDER BY, window) triggers this exchange.</text>
    <text x="36" y="348" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">Narrow transforms (filter, select, withColumn) never shuffle</text>
    <text x="36" y="378" style="font-size:20px;fill:#ffffff;font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica,Arial,sans-serif">because they don't need keys co-located.</text>
  </svg>
  <div class="legend">
    <span class="k"><span class="sw" style="background:#cfe8ec;border:1px solid #7fbcc8"></span>key = A</span>
    <span class="k"><span class="sw" style="background:#f7dcd0;border:1px solid #e0a882"></span>key = B</span>
    <span class="k"><span class="sw" style="background:#d6ecdd;border:1px solid #8bbf9c"></span>key = C</span>
    <span class="k"><span class="sw" style="background:#fdf3ec;border:1px solid #e5bfa2"></span>Network exchange</span>
  </div>
  <div class="note">In the Query Profile this appears as an <code>Exchange</code> operator between stages. The total bytes moved is <code>shuffle_read_bytes</code> in <code>system_vw.query.history</code>. This is where skew and spill originate: if key B has 80% of the rows, Task 1 gets 80% of the work.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The two phases of a shuffle
- **Write phase (map side)**: each task sorts its rows by `hash(key) mod N` and writes them to local shuffle files, one block per downstream partition. This is local disk I/O. On serverless with Photon, small shuffles stay entirely in memory.
- **Read phase (reduce side)**: each downstream task pulls the blocks it owns from every upstream task over the network. This is the expensive part. The bytes pulled are `shuffle_read_bytes` in `system_vw.query.history`, which equals `networkSentBytes` in the Query Profile.

### How many shuffle partitions?
- On serverless, AQE dynamically determines the number of post-shuffle partitions based on the actual data volume. You do not set `spark.sql.shuffle.partitions`.
- AQE coalesces small partitions into larger ones after the write phase completes, so you get fewer, right-sized tasks on the reduce side.
- On classic compute with AQE enabled (default since DBR 12.2), the same coalescing happens. The difference is that classic starts from the `spark.sql.shuffle.partitions` hint (default 200), then coalesces down. Serverless has no user-facing knob.

### Reading shuffle cost in the Query Profile
- Shuffles appear as **Exchange** operators between stages. Each Exchange is a stage boundary.
- Color the profile by **Memory peak** to identify which Exchanges buffer the most data. The task with the highest peak is the one closest to spilling.
- A query with three joins typically has at least four to six Exchanges (both sides of each sort-merge join). Broadcast joins eliminate Exchanges entirely.
- `shuffle_read_bytes` in `system_vw.query.history` is the single best metric for ranking which queries move the most data across your workload.

### Why shuffle is where the other three patterns show up
- **Skew**: if one key holds 80% of the rows, the task assigned that key's hash bucket gets 80% of the work. The other tasks finish and idle while the straggler runs alone.
- **Spill**: the reduce-side task buffers pulled blocks into a hash table (for hash joins) or sorts them (for sort-merge). If the working set exceeds available memory, it spills to local disk.
- **Explosion**: if a join produces far more output rows than input rows, the shuffle that follows the join carries the multiplied volume. A 10x fan-out at the join means 10x the shuffle bytes in the next Exchange.

### What you do NOT control with shuffle
- You do not reduce shuffle cost by hand-setting partition counts. What matters is the **join strategy** the optimizer selects: broadcast (no shuffle) vs sort-merge (shuffles both sides). The next section covers how it decides.
- You do not reduce shuffle cost by adding more compute. More tasks spread the work, but the total bytes moved stays the same. The effective fixes all reduce the volume that crosses the network: broadcast a small side, pre-aggregate before the join, or push filters down so fewer rows reach the shuffle.

</details>

### B1. How Spark Picks a Join Strategy
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-label { font-size: 14pt; font-weight: 700; letter-spacing: 2px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-row">
  <div class="ds-card green">
    <div class="ds-card-label">No shuffle</div>
    <div class="ds-card-title">Broadcast join</div>
    <ul>
      <li>Small side copied whole to every task, joined locally.</li>
      <li>Chosen when the small side is estimated under 10 MB.</li>
    </ul>
  </div>
  <div class="ds-card blue">
    <div class="ds-card-label">Shuffles both sides</div>
    <div class="ds-card-title">Sort-merge join</div>
    <ul>
      <li>Both sides redistributed by key, sorted, then merged.</li>
      <li>Default for large-to-large joins. Works at any scale.</li>
    </ul>
  </div>
  <div class="ds-card amber">
    <div class="ds-card-label">Shuffles both sides</div>
    <div class="ds-card-title">Shuffle hash join</div>
    <ul>
      <li>Same shuffle as sort-merge, but builds a hash table instead of sorting.</li>
      <li>Photon picks it when it estimates a cost advantage.</li>
    </ul>
  </div>
  <div class="ds-card red">
    <div class="ds-card-label">Last resort</div>
    <div class="ds-card-title">Broadcast nested loop</div>
    <ul>
      <li>No equality key to hash on. Every row compared to every row.</li>
      <li>Fix with a <code>RANGE_JOIN</code> hint and a bin size.</li>
    </ul>
  </div>
</div>
</div>

<div class="joinx">
  <style>
    .joinx { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .joinx h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .joinx p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .joinx svg { display: block; width: 100%; height: auto; }
    .joinx text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .joinx .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .joinx .sm { font-size: 12px; fill: #62707d; }
    .joinx .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .joinx .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .joinx code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Broadcast Join</h3>
  <p class="sub">The optimizer picks broadcast when one side is estimated under 10 MB. No rows cross the network.</p>
  <svg viewBox="0 0 900 240" role="img" aria-label="Broadcast join copies the small table to every task with no shuffle">
    <defs>
      <marker id="jn-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#00A972"></path>
      </marker>
      <marker id="jn-arr-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#4299E0"></path>
      </marker>
      <marker id="jn-arr-r" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#c2703a"></path>
      </marker>
    </defs>
    <!-- ========== TOP: BROADCAST JOIN ========== -->
    <rect x="0" y="0" width="900" height="230" rx="10" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="hdr" x="20" y="24">BROADCAST JOIN</text>
    <text class="sm" x="220" y="24">No shuffle. The small table is copied whole to every task.</text>
    <!-- Large table: stays in place -->
    <rect x="20" y="42" width="200" height="170" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="32" y="60" fill="#14343f">TABLE A (large)</text>
    <rect x="32" y="70" width="176" height="38" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="44" y="93">Task 0: partition 0</text>
    <rect x="32" y="114" width="176" height="38" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="44" y="137">Task 1: partition 1</text>
    <rect x="32" y="158" width="176" height="38" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="44" y="181">Task 2: partition 2</text>
    <!-- Small table -->
    <rect x="260" y="42" width="140" height="74" rx="8" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="tag" x="272" y="60" fill="#2d6e3f">TABLE B (small)</text>
    <text class="sm" x="272" y="78">4 rows (under 10 MB)</text>
    <text class="sm" x="272" y="93">Fits in memory</text>
    <text class="tag" x="272" y="108" fill="#2d6e3f">COPIED WHOLE</text>
    <!-- Arrows: small table copied to each task -->
    <line x1="400" y1="68" x2="470" y2="68" stroke="#00A972" stroke-width="1.6" marker-end="url(#jn-arr)"></line>
    <line x1="400" y1="80" x2="470" y2="112" stroke="#00A972" stroke-width="1.6" marker-end="url(#jn-arr)"></line>
    <line x1="400" y1="92" x2="470" y2="156" stroke="#00A972" stroke-width="1.6" marker-end="url(#jn-arr)"></line>
    <!-- Result: each task has both sides locally -->
    <rect x="480" y="42" width="220" height="170" rx="8" fill="#ffffff" stroke="#8bbf9c"></rect>
    <text class="tag" x="492" y="60" fill="#2d6e3f">EACH TASK JOINS LOCALLY</text>
    <rect x="492" y="70" width="196" height="38" rx="5" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="504" y="86">Task 0: large + small (local join)</text>
    <text class="tag" x="504" y="100" fill="#2d6e3f">JOIN locally</text>
    <rect x="492" y="114" width="196" height="38" rx="5" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="504" y="130">Task 1: large + small (local join)</text>
    <text class="tag" x="504" y="144" fill="#2d6e3f">JOIN locally</text>
    <rect x="492" y="158" width="196" height="38" rx="5" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="504" y="174">Task 2: large + small (local join)</text>
    <text class="tag" x="504" y="188" fill="#2d6e3f">JOIN locally</text>
    <!-- Result badge -->
    <rect x="730" y="80" width="150" height="72" rx="8" fill="#00A972" fill-opacity="0.15" stroke="#00A972"></rect>
    <text class="tag" x="746" y="104" fill="#2d6e3f">NO EXCHANGE</text>
    <text class="sm" x="746" y="120">No network cost</text>
    <text class="sm" x="746" y="134">on the join itself</text>
  </svg>
</div>

<div class="bj-tabs">
<style>
.bj-tabs { font-family: sans-serif; max-width: 1060px; margin: 18px 0 0; color: #0b2026; }
.bj-tabs * { box-sizing: border-box; }
.bj-tabs input[type="radio"] { display: none; }
.bj-tabs .bj-row { display: flex; gap: 16px; align-items: stretch; flex-wrap: wrap; }
.bj-tabs .bj-row > label { flex: 1; min-width: 240px; cursor: pointer; background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid var(--accent, #4299E0); padding: 18px 20px 16px; transition: box-shadow 0.2s, background 0.2s; }
.bj-tabs .bj-row > label:hover { box-shadow: 0 4px 14px rgba(27,49,57,0.12); }
.bj-tabs .bj-label { font-size: 12pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.bj-tabs .bj-title { font-size: 16pt; font-weight: 700; line-height: 1.3; margin: 0; }
.bj-tabs .bj-hint { font-size: 11pt; color: #618794; margin-top: 6px; }
.bj-tabs .bj-panel { display: none; width: 100%; background: #F9F7F4; border-radius: 0 0 8px 8px; border: 1px solid #EEEDE9; border-top: 3px solid var(--accent, #4299E0); padding: 18px 24px; margin-top: 12px; font-size: 13pt; line-height: 1.55; }
.bj-tabs .bj-panel code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 12pt; background: #f2f5f8; padding: 1px 5px; border-radius: 3px; color: #23424c; }
.bj-g{--accent:#00A972;} .bj-a{--accent:#FFAB00;} .bj-b{--accent:#4299E0;}
/* Active states */
#bj-t1:checked ~ .bj-row label[for="bj-t1"],
#bj-t2:checked ~ .bj-row label[for="bj-t2"],
#bj-t3:checked ~ .bj-row label[for="bj-t3"] { background: #ffffff; box-shadow: 0 4px 14px rgba(27,49,57,0.12); }
#bj-t1:checked ~ .bj-p1 { display: block; }
#bj-t2:checked ~ .bj-p2 { display: block; }
#bj-t3:checked ~ .bj-p3 { display: block; }
</style>

<input type="radio" name="bj-tab" id="bj-t1">
<input type="radio" name="bj-tab" id="bj-t2">
<input type="radio" name="bj-tab" id="bj-t3">

<div class="bj-row">
  <label for="bj-t1" class="bj-g">
    <div class="bj-label">Automatic</div>
    <div class="bj-title">Under 10 MB</div>
    <div class="bj-hint">Click to see details</div>
  </label>
  <label for="bj-t2" class="bj-a">
    <div class="bj-label">AQE runtime flip</div>
    <div class="bj-title">10 to 30 MB</div>
    <div class="bj-hint">Click to see details</div>
  </label>
  <label for="bj-t3" class="bj-b">
    <div class="bj-label">Developer override</div>
    <div class="bj-title">Over 30 MB</div>
    <div class="bj-hint">Click to see details</div>
  </label>
</div>

<div class="bj-panel bj-p1 bj-g">
  The planner checks estimated table size against <code>spark.sql.autoBroadcastJoinThreshold</code> (default <strong>10 MB</strong>). 
  
  If one side fits, it copies that side to every task and joins locally with no Exchange operator and no network cost. 
  
  The decision depends on accurate statistics, which Predictive Optimization keeps fresh on managed tables.
</div>

<div class="bj-panel bj-p2 bj-a">
  This is a separate, higher threshold because AQE works with measured bytes rather than an estimate, so it can be more aggressive. 
  
  After a shuffle stage completes, AQE checks the real output size against <code>spark.databricks.adaptive.autoBroadcastJoinThreshold</code> (default <strong>30 MB</strong>) and converts the sort-merge into a broadcast if it fits. 
  
  The shuffle network cost was already paid, so you only save the sort-merge compute, not the data movement.
</div>

<div class="bj-panel bj-p3 bj-b">
  You can force a broadcast join regardless of the size estimate using a join hint:
  <div style="display:flex;gap:12px;flex-wrap:wrap;margin:10px 0;">
    <div style="flex:1;min-width:220px;background:#f2f5f8;border-radius:6px;padding:10px 12px;">
      <div style="font-size:11pt;font-weight:700;color:#2272B4;margin-bottom:2px;">SQL</div>
      <code style="white-space:pre;font-size:12pt;">SELECT /*+ BROADCAST(dim) */ ...
FROM fact JOIN dim ON ...</code>
    </div>
    <div style="flex:1;min-width:220px;background:#f2f5f8;border-radius:6px;padding:10px 12px;">
      <div style="font-size:11pt;font-weight:700;color:#2272B4;margin-bottom:2px;">DataFrame</div>
      <code style="white-space:pre;font-size:12pt;">from pyspark.sql.functions import broadcast
fact.join(broadcast(dim), ...)</code>
    </div>
  </div>
  The planner broadcasts from the start, so no shuffle is ever scheduled. On serverless, <code>spark.sql.autoBroadcastJoinThreshold</code> is not settable (it raises <code>CONFIG_NOT_AVAILABLE</code>), so the hint is the only way to override the default.
  <br/><br/>
  <strong>Prefer fixing stats over adding hints.</strong> On managed tables, Predictive Optimization runs ANALYZE automatically so the planner gets accurate size estimates without intervention. Reserve the hint for cases where the optimizer cannot know the true size at plan time: complex subqueries, temporary views, or tables without fresh statistics.
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How the optimizer selects a strategy
- The planner checks the estimated size of each join side against `spark.sql.autoBroadcastJoinThreshold` (default 10 MB). If one side is small enough, it plans a broadcast join.
- If both sides exceed the threshold, it plans a sort-merge join (the safe default that works at any scale).
- Photon may convert a sort-merge to a shuffle hash join at plan time when its cost model estimates that building a per-partition hash table is cheaper than sorting. You cannot force this with a hint.
- If there is no equality condition at all (pure range, inequality, or cross join), the planner falls back to broadcast nested loop.

### The broadcast hint on serverless
- On serverless, `spark.sql.autoBroadcastJoinThreshold` is not available (`CONFIG_NOT_AVAILABLE`). The broadcast join hint (`/*+ BROADCAST(t) */` or `broadcast(df)`) is the developer override. It is always honored.
- In Demo 02, the 5M-row dimension tables were deliberately sized above the 10 MB threshold so the sort-merge join (and its shuffle cost) would be visible in the profile. A dimension just over the threshold can still end up broadcast by AQE at runtime (the 30 MB flip), and the benchmark had to account for that. Use the hint when you know your dimension is broadcastable and do not want to rely on the runtime estimate.

### AQE can still flip at runtime
- On Databricks, AQE adds a runtime switch governed by `spark.databricks.adaptive.autoBroadcastJoinThreshold` (default 30 MB). After a shuffle stage completes, AQE knows the real output size and can convert a planned sort-merge into a broadcast join if the real size is under that threshold.
- The catch: AQE can only flip after the first shuffle has run, so the shuffle cost was already paid. A statically planned broadcast (accurate stats, or a `BROADCAST` hint) avoids the shuffle entirely. This is one reason Predictive Optimization running `ANALYZE` on managed tables matters: good stats let the planner broadcast up front.

### Join reordering and range joins
- With column statistics present, the cost-based optimizer reorders inner joins to filter early and keep intermediate results small. Predictive Optimization runs `ANALYZE` automatically on UC managed tables, so the stats that drive reordering stay fresh without a cron.
- A range join (matching on `BETWEEN` or an interval, not equality) degenerates to a nested loop. Give it a `RANGE_JOIN` hint with a bin size so Spark can bucket the intervals instead of comparing every pair. The setup exposes `RANGE_JOIN_HINT_SQL` as an illustration.

</details>

<div class="joinx2">
  <style>
    .joinx2 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .joinx2 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .joinx2 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .joinx2 svg { display: block; width: 100%; height: auto; }
    .joinx2 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .joinx2 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .joinx2 .sm { font-size: 12px; fill: #62707d; }
    .joinx2 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .joinx2 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .joinx2 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Sort-Merge Join vs Shuffle Hash Join</h3>
  <p class="sub">Both shuffle both sides by key (2 Exchanges, same network cost). The difference is what happens after co-location.</p>
  <svg viewBox="0 0 900 490" role="img" aria-label="Two diagrams: sort-merge join sorts then merges, shuffle hash join builds a hash table instead of sorting">
    <defs>
      <marker id="jn2-arr-a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#FFAB00"></path>
      </marker>
      <marker id="jn2-arr-r" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#c2703a"></path>
      </marker>
      <marker id="jn2-arr-b" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#4299E0"></path>
      </marker>
    </defs>
    <!-- ========== TOP: SORT-MERGE JOIN ========== -->
    <rect x="0" y="0" width="900" height="230" rx="10" fill="#f0f6fd" stroke="#94bee0"></rect>
    <text class="hdr" x="20" y="24">SORT-MERGE JOIN</text>
    <text class="sm" x="180" y="24">Default for large-to-large. Both sides shuffled, sorted by key, then merged.</text>
    <!-- Table A -->
    <rect x="20" y="42" width="130" height="86" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="32" y="58" fill="#14343f">TABLE A (large)</text>
    <rect x="32" y="66" width="50" height="18" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="40" y="78" fill="#0f6b7a">k1</text>
    <rect x="86" y="66" width="50" height="18" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="94" y="78" fill="#7a4a2e">k2</text>
    <rect x="32" y="88" width="50" height="18" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="40" y="100" fill="#7a4a2e">k2</text>
    <rect x="86" y="88" width="50" height="18" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="94" y="100" fill="#0f6b7a">k1</text>
    <text class="sm" x="32" y="122">Keys mixed</text>
    <!-- Table B -->
    <rect x="20" y="140" width="130" height="72" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="32" y="156" fill="#14343f">TABLE B (large)</text>
    <rect x="32" y="164" width="50" height="18" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="40" y="176" fill="#0f6b7a">k1</text>
    <rect x="86" y="164" width="50" height="18" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="94" y="176" fill="#7a4a2e">k2</text>
    <text class="sm" x="32" y="200">Keys mixed</text>
    <!-- Exchange 1 -->
    <rect x="176" y="42" width="110" height="86" rx="8" fill="#fef8f4" stroke="#e5bfa2"></rect>
    <text class="tag" x="190" y="58" fill="#7a4a2e">EXCHANGE 1</text>
    <text class="sm" x="190" y="74">Shuffle A</text>
    <text class="sm" x="190" y="88">by join key</text>
    <line x1="150" y1="82" x2="176" y2="82" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <!-- Exchange 2 -->
    <rect x="176" y="140" width="110" height="72" rx="8" fill="#fef8f4" stroke="#e5bfa2"></rect>
    <text class="tag" x="190" y="156" fill="#7a4a2e">EXCHANGE 2</text>
    <text class="sm" x="190" y="172">Shuffle B</text>
    <text class="sm" x="190" y="186">by join key</text>
    <line x1="150" y1="176" x2="176" y2="176" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <!-- Co-located -->
    <rect x="312" y="42" width="170" height="170" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="324" y="58" fill="#14343f">CO-LOCATED BY KEY</text>
    <rect x="324" y="66" width="146" height="42" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="336" y="82">Task 0: all k1 from A</text>
    <text class="sm" x="336" y="96">Task 0: all k1 from B</text>
    <rect x="324" y="114" width="146" height="42" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="336" y="130">Task 1: all k2 from A</text>
    <text class="sm" x="336" y="144">Task 1: all k2 from B</text>
    <text class="sm" x="324" y="192">Sorted and merged per task</text>
    <!-- Arrows exchange to co-located -->
    <line x1="286" y1="82" x2="312" y2="82" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <line x1="286" y1="176" x2="312" y2="140" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <!-- Join step: sort + merge -->
    <rect x="510" y="42" width="190" height="170" rx="8" fill="#f0f6fd" stroke="#4299E0"></rect>
    <text class="tag" x="524" y="58" fill="#1B5162">SORT + MERGE</text>
    <rect x="524" y="68" width="164" height="56" rx="5" fill="#e8f1fb" stroke="#94bee0"></rect>
    <text class="sm" x="536" y="84">Both sides sorted by key</text>
    <text class="sm" x="536" y="98">within each partition</text>
    <text class="sm" x="536" y="116">&#8595;</text>
    <rect x="524" y="126" width="164" height="56" rx="5" fill="#e8f1fb" stroke="#94bee0"></rect>
    <text class="sm" x="536" y="142">Merged in one pass</text>
    <text class="sm" x="536" y="156">(two sorted lists)</text>
    <text class="tag" x="524" y="196" fill="#1B5162">SCALES TO ANY SIZE</text>
    <!-- Arrow co-located to sort-merge -->
    <line x1="482" y1="120" x2="510" y2="120" stroke="#4299E0" stroke-width="1.6" marker-end="url(#jn2-arr-b)"></line>
    <!-- Badge -->
    <rect x="720" y="60" width="160" height="72" rx="8" fill="#4299E0" fill-opacity="0.1" stroke="#4299E0"></rect>
    <text class="tag" x="736" y="84" fill="#1B5162">2 EXCHANGES</text>
    <text class="sm" x="736" y="100">Both sides cross the</text>
    <text class="sm" x="736" y="114">network. Default when</text>
    <text class="sm" x="736" y="128">both sides are large.</text>
    <!-- ========== BOTTOM: SHUFFLE HASH JOIN ========== -->
    <rect x="0" y="250" width="900" height="230" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="hdr" x="20" y="274">SHUFFLE HASH JOIN</text>
    <text class="sm" x="190" y="274">Same shuffle as above, but builds a hash table instead of sorting.</text>
    <!-- Table A -->
    <rect x="20" y="292" width="130" height="86" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="32" y="308" fill="#14343f">TABLE A (large)</text>
    <rect x="32" y="316" width="50" height="18" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="40" y="328" fill="#0f6b7a">k1</text>
    <rect x="86" y="316" width="50" height="18" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="94" y="328" fill="#7a4a2e">k2</text>
    <rect x="32" y="338" width="50" height="18" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="40" y="350" fill="#7a4a2e">k2</text>
    <rect x="86" y="338" width="50" height="18" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="94" y="350" fill="#0f6b7a">k1</text>
    <text class="sm" x="32" y="372">Keys mixed</text>
    <!-- Table B -->
    <rect x="20" y="390" width="130" height="72" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="32" y="406" fill="#14343f">TABLE B (large)</text>
    <rect x="32" y="414" width="50" height="18" rx="4" fill="#cfe8ec" stroke="#7fbcc8"></rect>
    <text class="tag" x="40" y="426" fill="#0f6b7a">k1</text>
    <rect x="86" y="414" width="50" height="18" rx="4" fill="#f7dcd0" stroke="#e0a882"></rect>
    <text class="tag" x="94" y="426" fill="#7a4a2e">k2</text>
    <text class="sm" x="32" y="450">Keys mixed</text>
    <!-- Exchange 1 -->
    <rect x="176" y="292" width="110" height="86" rx="8" fill="#fef8f4" stroke="#e5bfa2"></rect>
    <text class="tag" x="190" y="308" fill="#7a4a2e">EXCHANGE 1</text>
    <text class="sm" x="190" y="326">Shuffle A</text>
    <text class="sm" x="190" y="340">by join key</text>
    <line x1="150" y1="332" x2="176" y2="332" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <!-- Exchange 2 -->
    <rect x="176" y="390" width="110" height="72" rx="8" fill="#fef8f4" stroke="#e5bfa2"></rect>
    <text class="tag" x="190" y="406" fill="#7a4a2e">EXCHANGE 2</text>
    <text class="sm" x="190" y="424">Shuffle B</text>
    <text class="sm" x="190" y="438">by join key</text>
    <line x1="150" y1="426" x2="176" y2="426" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <!-- Co-located -->
    <rect x="312" y="292" width="170" height="170" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="324" y="308" fill="#14343f">CO-LOCATED BY KEY</text>
    <rect x="324" y="316" width="146" height="42" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="336" y="332">Task 0: all k1 from A</text>
    <text class="sm" x="336" y="346">Task 0: all k1 from B</text>
    <rect x="324" y="364" width="146" height="42" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="336" y="380">Task 1: all k2 from A</text>
    <text class="sm" x="336" y="394">Task 1: all k2 from B</text>
    <text class="sm" x="324" y="442">Same co-location as above</text>
    <!-- Arrows exchange to co-located -->
    <line x1="286" y1="332" x2="312" y2="332" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <line x1="286" y1="426" x2="312" y2="390" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#jn2-arr-r)"></line>
    <!-- The difference: hash table + probe -->
    <rect x="510" y="282" width="190" height="180" rx="8" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="tag" x="524" y="300" fill="#7a4a2e">HASH TABLE + PROBE</text>
    <rect x="524" y="310" width="164" height="56" rx="5" fill="#fff5d6" stroke="#e5c766"></rect>
    <text class="sm" x="536" y="328">Smaller side: loaded</text>
    <text class="sm" x="536" y="344">into hash table</text>
    <text class="sm" x="536" y="358">&#8595;</text>
    <rect x="524" y="368" width="164" height="56" rx="5" fill="#fff5d6" stroke="#e5c766"></rect>
    <text class="sm" x="536" y="386">Larger side: each row</text>
    <text class="sm" x="536" y="402">finds its match instantly</text>
    <text class="tag" x="524" y="446" fill="#7a4a2e">NO SORT STEP</text>
    <!-- Arrow co-located to hash -->
    <line x1="482" y1="362" x2="510" y2="362" stroke="#FFAB00" stroke-width="1.6" marker-end="url(#jn2-arr-a)"></line>
    <!-- Badge -->
    <rect x="720" y="310" width="160" height="72" rx="8" fill="#FFAB00" fill-opacity="0.12" stroke="#FFAB00"></rect>
    <text class="tag" x="736" y="334" fill="#7a4a2e">2 EXCHANGES</text>
    <text class="sm" x="736" y="350">Same shuffle cost,</text>
    <text class="sm" x="736" y="364">faster join step</text>
    <text class="sm" x="736" y="378">(no sort needed)</text>
  </svg>
  <div class="note"><strong>When the optimizer picks shuffle hash over sort-merge:</strong> Photon may convert a planned sort-merge to shuffle hash at plan time when its cost model estimates that building a per-partition hash table is cheaper than sorting both sides. You cannot force this with a hint; the optimizer decides based on table sizes and the Photon cost model. Both strategies produce the same Exchange operators and shuffle bytes in the profile.</div>
</div>

<div class="joinx3">
  <style>
    .joinx3 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .joinx3 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .joinx3 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .joinx3 svg { display: block; width: 100%; height: auto; }
    .joinx3 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .joinx3 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .joinx3 .sm { font-size: 12px; fill: #62707d; }
    .joinx3 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .joinx3 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .joinx3 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Broadcast Nested Loop</h3>
  <p class="sub">The only option when the join has no equality condition (range, inequality, or cross). The small side is broadcast, but every row is compared to every row.</p>
  <svg viewBox="0 0 900 240" role="img" aria-label="Broadcast nested loop: small side broadcast to every task, then every row compared against every row in a nested loop">
    <defs>
      <marker id="jn3-arr-red" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#98102A"></path>
      </marker>
    </defs>
    <rect x="0" y="0" width="900" height="230" rx="10" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="hdr" x="20" y="24">BROADCAST NESTED LOOP</text>
    <text class="sm" x="230" y="24">No equality key to hash on. Every row compared to every row.</text>
    <!-- Table A: large, stays -->
    <rect x="20" y="42" width="180" height="140" rx="8" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="32" y="58" fill="#14343f">TABLE A (large)</text>
    <rect x="32" y="66" width="156" height="30" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="44" y="85">Task 0: partition 0</text>
    <rect x="32" y="100" width="156" height="30" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="44" y="119">Task 1: partition 1</text>
    <rect x="32" y="134" width="156" height="30" rx="5" fill="#f7fafc" stroke="#d3dde5"></rect>
    <text class="sm" x="44" y="153">Task 2: partition 2</text>
    <!-- Table B: small, broadcast -->
    <rect x="230" y="42" width="140" height="74" rx="8" fill="#fde0e4" stroke="#d4808f"></rect>
    <text class="tag" x="242" y="58" fill="#98102A">TABLE B (small)</text>
    <text class="sm" x="242" y="76">Copied to every task</text>
    <text class="sm" x="242" y="90">No equality key</text>
    <text class="tag" x="242" y="108" fill="#98102A">COPIED WHOLE</text>
    <!-- Arrows: B copied to each task -->
    <line x1="370" y1="60" x2="420" y2="60" stroke="#98102A" stroke-width="1.4" marker-end="url(#jn3-arr-red)"></line>
    <line x1="370" y1="76" x2="420" y2="100" stroke="#98102A" stroke-width="1.4" marker-end="url(#jn3-arr-red)"></line>
    <line x1="370" y1="92" x2="420" y2="140" stroke="#98102A" stroke-width="1.4" marker-end="url(#jn3-arr-red)"></line>
    <!-- Nested loop comparison -->
    <rect x="430" y="42" width="230" height="170" rx="8" fill="#ffffff" stroke="#d4808f"></rect>
    <text class="tag" x="442" y="58" fill="#98102A">NESTED LOOP PER TASK</text>
    <rect x="442" y="66" width="206" height="56" rx="5" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="sm" x="454" y="82">For each row of A, compare</text>
    <text class="sm" x="454" y="96">against EVERY row of B</text>
    <text class="sm" x="454" y="112">Condition: range or inequality</text>
    <rect x="442" y="130" width="206" height="44" rx="5" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="sm" x="454" y="146">100 A rows x 50 B rows</text>
    <text class="sm" x="454" y="162">= 5,000 comparisons per task</text>
    <text class="tag" x="442" y="198" fill="#98102A">O(N x M) PER TASK</text>
    <!-- Badge -->
    <rect x="690" y="50" width="190" height="100" rx="8" fill="#98102A" fill-opacity="0.1" stroke="#98102A"></rect>
    <text class="tag" x="706" y="72" fill="#98102A">NO EXCHANGE, BUT</text>
    <text class="tag" x="706" y="88" fill="#98102A">QUADRATIC COST</text>
    <text class="sm" x="706" y="106">2x rows on one side =</text>
    <text class="sm" x="706" y="120">4x total comparisons</text>
    <text class="sm" x="706" y="140" style="fill:#98102A;font-weight:600">Fix: RANGE_JOIN + bin size</text>
  </svg>
  <div class="note"><strong>When you see this:</strong> Broadcast nested loop is not a fallback the optimizer reaches after trying other strategies. It is the only strategy available when the join condition has no equality key (pure range, inequality, or cross join). The other three strategies all need an equality key to hash or sort on.<br/><br/>Common real-world cases include interval-overlap joins (sessionization, billing windows) and range-predicate joins.<br/><br/>It is quadratic: doubling rows on one side quadruples the total comparisons. Give a range join a <code>/*+ RANGE_JOIN(table, bin_size) */</code> hint so Spark can bucket the intervals instead of comparing every pair.</div>
</div>

### B2. Run a Shuffle Hash Join and a Broadcast Join

This 4-table join aggregates revenue by customer tier and product category. The dimension tables (**customers**, **products**) are well above the 10 MB broadcast threshold, so every join **shuffles both sides** through Exchange operators. Photon typically selects **shuffle hash join** here (hash table + probe, no sort step).

Run the next cell, then open its Query Profile (**See performance** under the cell, or **Query History** then **See query profile**) and color the graph by **Memory peak**.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  c.tier,
  p.category,
  SUM(oi.quantity * p.list_price) AS revenue,
  COUNT(*) AS line_items
FROM orders o
JOIN order_items oi ON o.order_idx = oi.order_idx
JOIN products p ON oi.product_idx = p.product_idx
JOIN customers c ON o.customer_idx = c.customer_idx
GROUP BY c.tier, p.category
ORDER BY revenue DESC
</pre>
</div>

In [ ]:
# .count() forces full execution so the Query Profile has something to show.
print("Running SHUFFLE_JOIN_SQL (open its Query Profile after this cell) ...")
_shuffle_rows = spark.sql(SHUFFLE_JOIN_SQL).count()
print(f"Returned {_shuffle_rows:,} grouped rows. Look for Exchange operators and shuffle bytes.")

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info { --accent:#1B5162; --tint:#F8F9FC; }
</style>

<div class="ds-callout info">
  <div class="ds-eyebrow">Open the Query Profile</div>
  <div class="ds-callout-title">What to look for in this shuffle join</div>
  <ul>
    <li><strong>Join algorithm: Photon Shuffled Hash</strong>: Photon picked shuffle hash join over sort-merge because it estimated that building a per-partition hash table is cheaper than sorting. This is the strategy from the diagram above (same 2 Exchanges, but hash table + probe instead of sort + merge).</li>
    <li><strong>Exchange operators</strong>: you should see multiple Exchanges in the graph. Each join shuffles both sides, so 3 joins can produce up to 6 Exchanges (AQE may coalesce some).</li>
    <li><strong>Color by Memory peak</strong>: the Exchange operators light up because they buffer the full shuffle output in memory before passing it to the hash build step.</li>
    <li><strong>Rows flowing through joins</strong>: orders (10M) joins order_items (60M), producing ~60M rows that then flow into the products and customers joins. The row count at the first join output is the largest in the graph.</li>
    <li><strong>No Sort operators</strong>: unlike sort-merge, Photon Shuffled Hash skips the sort. You will NOT see Sort nodes between the Exchanges and the joins. This is the cost advantage Photon detected.</li>
    <li><strong>The final Aggregate</strong>: after all joins, the GROUP BY collapses ~60M rows down to 18 grouped rows. The shuffle cost was incurred to produce a tiny result.</li>
  </ul>
</div>
</div>

### B3. The Same Fact Table, Joined to a Tiny Lookup

Now join **orders** to the 4-row **channel_dim**. It is far under the 10 MB broadcast threshold, so the optimizer broadcasts it and shuffles nothing on the join. Run the next cell, then compare this profile to the one above: the join has no Exchange.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  ch.channel_label,
  COUNT(*) AS orders
FROM orders o
JOIN channel_dim ch ON o.channel = ch.channel
GROUP BY ch.channel_label
ORDER BY orders DESC
</pre>
</div>

In [ ]:
print("Running BROADCAST_JOIN_SQL (open its Query Profile after this cell) ...")
_broadcast_rows = spark.sql(BROADCAST_JOIN_SQL).count()
print(f"Returned {_broadcast_rows:,} grouped rows. The join broadcasts channel_dim, so no shuffle on the join.")

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.green { --accent:#00A972; --tint:#f0faf5; }
</style>

<div class="ds-callout green">
  <div class="ds-eyebrow">Open the Query Profile</div>
  <div class="ds-callout-title">What to look for in this broadcast join</div>
  <ul>
    <li><strong>No Exchange on the join</strong>: the join operator has no Exchange feeding either side. <strong>channel_dim</strong> (4 rows) was broadcast to every task, so rows stayed local. Compare this to the shuffle hash join above where every join had Exchange operators on both sides.</li>
    <li><strong>Join algorithm: Broadcast Hash Join</strong>: the profile labels the join as a broadcast hash join. Each task builds a hash table from the 4 broadcast rows and probes it with its local orders partition.</li>
    <li><strong>The GROUP BY shuffle is negligible</strong>: there is technically an Exchange for the aggregation, but it moves only 28 rows (the partial aggregates from each task, 4 groups each). <code>networkSentBytes</code> registers as 0.</li>
    <li><strong>Scan dominates</strong>: with no join shuffle, reading the 10M orders rows is the costliest operation (100% of time). The join itself takes a fraction of that.</li>
    <li><strong>Zero network shuffle</strong>: <code>shuffle_read_bytes</code> is 0 compared to 779 MB for the shuffle hash join above. The broadcast eliminated the expensive data movement entirely. Only 9 tasks were needed (vs 140).</li>
  </ul>
</div>
</div>

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.info { --accent:#1B5162; --tint:#F8F9FC; }
</style>

<div class="ds-callout info">
  <div class="ds-eyebrow">What to compare</div>
  <div class="ds-callout-title">Two joins, two very different profiles</div>
  <ul>
    <li>The shuffle hash join moves hundreds of megabytes across the network. Its graph has Exchange operators on both sides of each join, and shuffle dominates the memory-peak view.</li>
    <li>The broadcast join moves nothing on the join. <strong>channel_dim</strong> is copied to every task, so there is no Exchange feeding the join. <code>networkSentBytes</code> is 0.</li>
    <li>The optimizer made this decision based on table size estimates.</li>
    <li>The 5M-row dimension tables exceeded the 10 MB broadcast threshold, so they needed to shuffle.</li>
    <li>The 4-row lookup fit easily, so it was broadcasted.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Reading the two profiles side by side
- **Shuffle hash (SHUFFLE_JOIN_SQL)**: three joins across **orders**, **order_items**, **products**, and **customers**. The dimension tables are far above the 10 MB threshold, so each join shuffles both sides. Photon picks shuffle hash join (no sort step). Color by **Memory peak** and the Exchange operators light up. This is the common shape of a fact-to-large-dimension join.
- **Broadcast (BROADCAST_JOIN_SQL)**: **channel_dim** has four rows. The optimizer broadcasts it to every task, so the join needs no redistribution. The `GROUP BY` still shuffles, but the join itself does not. `shuffle_read_bytes` is dramatically lower.
- The exact bytes and task counts vary between runs. What is stable is the presence or absence of the Exchange on the join, which is what we want to see.
</details>

## C. Skew
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.caution { --bnr-bg: rgba(255,171,0,0.12); --bnr-accent: #FFAB00; --bnr-icon-fg: #0b2026; --bnr-title: #0b2026; --bnr-tagline: #5E7077; }
.flow-grid { display: grid; grid-template-columns: 1fr auto 1fr auto 1fr; gap: 0; align-items: stretch; }
.flow-step { background: #F9F7F4; border: 3px solid var(--accent, #4299E0); border-radius: 10px; padding: 16px 12px; box-shadow: 0 2px 8px rgba(27,49,57,0.08); text-align: center; min-width: 0; }
.flow-num { font-size: 26pt; font-weight: 800; color: var(--accent, #4299E0); line-height: 1; margin-bottom: 8px; }
.flow-label { font-size: 14pt; font-weight: 600; color: #0b2026; line-height: 1.35; }
.flow-arrow { display: flex; align-items: center; justify-content: center; color: #618794; font-size: 22pt; line-height: 1; padding: 0 10px; }
</style>

<div class="ds-banner caution">
  <div class="ds-banner-icon">!</div>
  <div>
    <div class="ds-banner-title">Skew is one task doing all the work</div>
    <div class="ds-banner-tagline">A few hot keys own most of the rows, so their partition runs long after the others finish. Fix cheapest first: AQE, then key cleanup, then salt.</div>
  </div>
</div>

<div class="flow-grid">
  <div class="flow-step" style="--accent:#00A972;"><div class="flow-num">1</div><div class="flow-label">Let AQE handle it<br/>(already on)</div></div>
  <div class="flow-arrow">&rarr;</div>
  <div class="flow-step" style="--accent:#4299E0;"><div class="flow-num">2</div><div class="flow-label">Filter null and<br/>junk keys</div></div>
  <div class="flow-arrow">&rarr;</div>
  <div class="flow-step" style="--accent:#618794; border-style:dashed; opacity:0.55;"><div class="flow-num">3</div><div class="flow-label">Salt the hot key<br/>(last resort)</div></div>
</div>

<div style="border-left:4px solid #618794;background:rgba(97,135,148,0.06);padding:12px 18px;border-radius:4px;max-width:1100px;margin:12px auto 0;font-family:sans-serif;">
  <strong style="color:#618794;font-size:14pt;">Why salting is costly</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.5;margin-top:4px;">Salting adds an extra shuffle and sort, replicates the small side across all salt buckets, and makes the query harder to read and maintain. On serverless, where AQE handles most skew automatically, salting is rarely justified. Use it only after AQE and the null/junk-key filter are both exhausted and you are prepared to pay that complexity cost.</div>
</div>

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.12);padding:14px 18px;border-radius:4px;">
  <strong style="display:block;color:#0b2026;margin-bottom:6px;font-size:15pt;">Confirm skew before you fix it</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">The <strong>DATA_SKEW</strong> insight names it for you: "Data is distributed unevenly across computing resources." In the profile, color by <strong>Time</strong> and look for one task in a Shuffle or Join running far longer than the rest.</div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What skew looks like
- Uneven key distribution leaves a few cores overloaded while the rest sit idle. In the profile you see it as one straggler task: the stage's max task time is many times its median. The **DATA_SKEW** insight flags this automatically.
- In our data, about 80% of orders belong to about 1% of customers (the whales). Any aggregation or join on `customer_idx` concentrates those rows onto a few tasks.

### The fix order, cheapest first
- **Step 1, let AQE handle it.** AQE's skew-join handling is on by default. When a shuffle partition is both larger than 256 MB and more than 5 times the median partition size, AQE splits it into sub-partitions and runs them in parallel. For sort-merge and shuffle-hash joins this often fixes skew with zero code change. Always check whether it already did before doing anything else.
- **Step 2, fix the keys.** The classic culprit is null or junk join keys piling onto one partition. Filter them out, or pre-aggregate, before the join. This is a one-line change and it is the next thing to try.
- **Step 3, salt.** Only after AQE and key fixes are exhausted. Salting spreads a hot key across N buckets by appending a random salt, joining on key plus salt, then combining. It works, but it is a real code change and it complicates the query, so it is the last resort.

</details>

<div class="skewx">
  <style>
    .skewx { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .skewx h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .skewx p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .skewx svg { display: block; width: 100%; height: auto; }
    .skewx text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .skewx .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .skewx .sm { font-size: 12px; fill: #62707d; }
    .skewx .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .skewx .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #FFAB00; padding: 4px 0 4px 10px; line-height: 1.55; }
    .skewx code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>What Skew Looks Like</h3>
  <p class="sub">One task holds a disproportionate share of rows. It runs long after the others finish.</p>
  <svg viewBox="0 0 900 290" role="img" aria-label="Diagram showing data skew: one task receives most of the rows after a shuffle, creating a straggler">
    <!-- Left: data distribution after shuffle -->
    <rect x="0" y="0" width="420" height="280" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="hdr" x="20" y="24">AFTER SHUFFLE (data per task)</text>
    <!-- Task bars -->
    <text class="sm" x="30" y="56">Task 0</text>
    <rect x="100" y="44" width="60" height="18" rx="4" fill="#00A972"></rect>
    <text class="sm" x="166" y="56">2%</text>
    <text class="sm" x="30" y="82">Task 1</text>
    <rect x="100" y="70" width="50" height="18" rx="4" fill="#00A972"></rect>
    <text class="sm" x="156" y="82">1.5%</text>
    <text class="sm" x="30" y="108">Task 2</text>
    <rect x="100" y="96" width="290" height="18" rx="4" fill="#FFAB00"></rect>
    <text class="tag" x="166" y="108" fill="#7a4a2e">80% (hot key!)</text>
    <text class="sm" x="30" y="134">Task 3</text>
    <rect x="100" y="122" width="70" height="18" rx="4" fill="#00A972"></rect>
    <text class="sm" x="176" y="134">3%</text>
    <text class="sm" x="30" y="160">Task 4</text>
    <rect x="100" y="148" width="55" height="18" rx="4" fill="#00A972"></rect>
    <text class="sm" x="161" y="160">2%</text>
    <!-- Straggler annotation -->
    <rect x="30" y="180" width="370" height="80" rx="6" fill="#ffffff" stroke="#e5c766"></rect>
    <text class="tag" x="44" y="200" fill="#7a4a2e">STRAGGLER</text>
    <text class="sm" x="44" y="218">Tasks 0,1,3,4 finish in 2s.</text>
    <text class="sm" x="44" y="234">Task 2 takes 45s (all other tasks idle).</text>
    <text class="sm" x="44" y="250">Stage wall-clock = slowest task = 45s.</text>
    <!-- Right: profile view -->
    <rect x="440" y="0" width="460" height="280" rx="10" fill="#F8F9FC" stroke="#94bee0"></rect>
    <text class="hdr" x="460" y="24">WHAT YOU SEE IN THE PROFILE</text>
    <!-- Task time bars (horizontal) -->
    <text class="tag" x="460" y="52" fill="#1B5162">COLOR BY TIME</text>
    <rect x="460" y="60" width="30" height="14" rx="3" fill="#00A972"></rect>
    <text class="sm" x="496" y="71">Task 0: 2s</text>
    <rect x="460" y="78" width="25" height="14" rx="3" fill="#00A972"></rect>
    <text class="sm" x="491" y="89">Task 1: 1.8s</text>
    <rect x="460" y="96" width="380" height="14" rx="3" fill="#FFAB00"></rect>
    <text class="tag" x="496" y="107" fill="#7a4a2e">Task 2: 45s</text>
    <rect x="460" y="114" width="35" height="14" rx="3" fill="#00A972"></rect>
    <text class="sm" x="501" y="125">Task 3: 2.5s</text>
    <rect x="460" y="132" width="28" height="14" rx="3" fill="#00A972"></rect>
    <text class="sm" x="494" y="143">Task 4: 2.1s</text>
    <!-- Insight badge -->
    <rect x="460" y="160" width="420" height="100" rx="8" fill="#FFAB00" fill-opacity="0.12" stroke="#FFAB00"></rect>
    <text class="tag" x="476" y="182" fill="#7a4a2e">PERFORMANCE INSIGHT: DATA_SKEW</text>
    <text class="sm" x="476" y="200">"Data is distributed unevenly across</text>
    <text class="sm" x="476" y="214">computing resources."</text>
    <text class="sm" x="476" y="236">Fires when one partition is larger than</text>
    <text class="sm" x="476" y="250">256 MB AND more than 5x the median.</text>
  </svg>
  <div class="note">In our demo data, 80% of orders belong to 1% of customers (the whales). Any aggregation or join on <code>customer_idx</code> concentrates those rows onto a few tasks, producing this exact pattern.</div>
</div>

### C1. The Whale Skew

A few customers own most of the orders. Run the aggregation, open the profile, and color by **Time**.

AQE's skew-join handling is already on, so for a plain aggregation like this the platform absorbs most of the imbalance for you. The point is to see the distribution and confirm the **DATA_SKEW** insight.

The whale customers (1% of IDs owning 80% of rows) land on a few partitions after the GROUP BY shuffle.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  o.customer_idx,
  COUNT(*) AS order_count
FROM orders o
GROUP BY o.customer_idx
ORDER BY order_count DESC
LIMIT 20
</pre>
</div>

In [ ]:
print("Running SKEW_WHALE_SQL (open its Query Profile, color by Time) ...")
_whale_rows = spark.sql(SKEW_WHALE_SQL).count()
print(f"Returned top {_whale_rows} customers by order count. A few whales dominate.")

### C2. Null and Junk Keys

**order_items_dirty** carries about 15% rows with a null **order_idx** (a broken upstream extract). We LEFT JOIN it to **orders**, which preserves those null-key rows, so they all shuffle to one partition and create a straggler, then get discarded. Run it and look for the one long task.

<div class="skew2">
  <style>
    .skew2 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .skew2 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .skew2 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .skew2 svg { display: block; width: 100%; height: auto; }
    .skew2 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .skew2 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .skew2 .sm { font-size: 12px; fill: #62707d; }
    .skew2 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .skew2 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #4299E0; padding: 4px 0 4px 10px; line-height: 1.55; }
    .skew2 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Why Null Keys Cause Skew</h3>
  <p class="sub">All null keys hash to the same value, so they all land on one partition after the shuffle.</p>
  <svg viewBox="0 0 900 300" role="img" aria-label="Diagram showing how null join keys all hash to the same partition, creating a straggler task">
    <defs>
      <marker id="sk2-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#c2703a"></path>
      </marker>
      <marker id="sk2-arr-g" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#00A972"></path>
      </marker>
    </defs>
    <!-- Left: source rows -->
    <rect x="0" y="0" width="200" height="290" rx="10" fill="#F8F9FC" stroke="#94bee0"></rect>
    <text class="hdr" x="20" y="24">SOURCE ROWS</text>
    <text class="sm" x="20" y="44">(order_items_dirty)</text>
    <rect x="20" y="56" width="160" height="22" rx="4" fill="#e8f1fb" stroke="#94bee0"></rect>
    <text class="sm" x="32" y="71">order_idx = 1001</text>
    <rect x="20" y="82" width="160" height="22" rx="4" fill="#e8f1fb" stroke="#94bee0"></rect>
    <text class="sm" x="32" y="97">order_idx = 2045</text>
    <rect x="20" y="108" width="160" height="22" rx="4" fill="#e8f1fb" stroke="#94bee0"></rect>
    <text class="sm" x="32" y="123">order_idx = 3782</text>
    <rect x="20" y="140" width="160" height="22" rx="4" fill="#fde0e4" stroke="#d4808f"></rect>
    <text class="sm" x="32" y="155" style="fill:#98102A">order_idx = NULL</text>
    <rect x="20" y="166" width="160" height="22" rx="4" fill="#fde0e4" stroke="#d4808f"></rect>
    <text class="sm" x="32" y="181" style="fill:#98102A">order_idx = NULL</text>
    <rect x="20" y="192" width="160" height="22" rx="4" fill="#fde0e4" stroke="#d4808f"></rect>
    <text class="sm" x="32" y="207" style="fill:#98102A">order_idx = NULL</text>
    <rect x="20" y="218" width="160" height="22" rx="4" fill="#fde0e4" stroke="#d4808f"></rect>
    <text class="sm" x="32" y="233" style="fill:#98102A">order_idx = NULL</text>
    <text class="tag" x="20" y="260" fill="#98102A">~15% OF ROWS ARE NULL</text>
    <text class="sm" x="20" y="278">Broken upstream extract</text>
    <line x1="200" y1="90" x2="240" y2="90" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#sk2-arr)"></line>
    <line x1="200" y1="185" x2="240" y2="210" stroke="#98102A" stroke-width="2" marker-end="url(#sk2-arr)"></line>
    <rect x="240" y="60" width="140" height="80" rx="8" fill="#fef8f4" stroke="#e5bfa2"></rect>
    <text class="tag" x="260" y="80" fill="#7a4a2e">EXCHANGE</text>
    <text class="sm" x="260" y="98">hash(order_idx)</text>
    <text class="sm" x="260" y="114">determines partition</text>
    <rect x="240" y="170" width="140" height="60" rx="8" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="tag" x="260" y="190" fill="#98102A">hash(NULL)</text>
    <text class="sm" x="260" y="208">Always same value!</text>
    <text class="sm" x="260" y="222">All NULLs → 1 task</text>
    <line x1="380" y1="90" x2="430" y2="70" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#sk2-arr)"></line>
    <line x1="380" y1="100" x2="430" y2="110" stroke="#c2703a" stroke-width="1.4" stroke-dasharray="4 3" marker-end="url(#sk2-arr)"></line>
    <line x1="380" y1="200" x2="430" y2="200" stroke="#98102A" stroke-width="2" marker-end="url(#sk2-arr)"></line>
    <rect x="430" y="0" width="220" height="290" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="hdr" x="450" y="24">TASKS AFTER SHUFFLE</text>
    <rect x="450" y="42" width="180" height="40" rx="6" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="sm" x="462" y="58">Task 0: hash=0 rows</text>
    <text class="tag" x="462" y="74" fill="#00A972">FAST (few rows)</text>
    <rect x="450" y="88" width="180" height="40" rx="6" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="sm" x="462" y="104">Task 1: hash=1 rows</text>
    <text class="tag" x="462" y="120" fill="#00A972">FAST (few rows)</text>
    <rect x="450" y="148" width="180" height="72" rx="6" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="sm" x="462" y="166">Task 2: ALL null rows</text>
    <text class="tag" x="462" y="184" fill="#98102A">STRAGGLER</text>
    <text class="sm" x="462" y="202">15% of entire table</text>
    <text class="sm" x="462" y="216">on one task</text>
    <text class="sm" x="450" y="250">None of these null rows will</text>
    <text class="sm" x="450" y="264">match in the join anyway.</text>
    <text class="sm" x="450" y="278">They just waste time.</text>
    <rect x="670" y="0" width="230" height="290" rx="10" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="hdr" x="690" y="24">THE FIX</text>
    <text class="tag" x="690" y="50" fill="#00A972">FILTER BEFORE THE JOIN</text>
    <rect x="690" y="62" width="190" height="50" rx="6" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="702" y="80">WHERE order_idx</text>
    <text class="sm" x="702" y="96">IS NOT NULL</text>
    <text class="sm" x="690" y="130">Removes the hot partition</text>
    <text class="sm" x="690" y="146">entirely. One line of SQL.</text>
    <text class="sm" x="690" y="170">The null rows can never</text>
    <text class="sm" x="690" y="186">match in the join (NULL</text>
    <text class="sm" x="690" y="202">never equals anything),</text>
    <text class="sm" x="690" y="218">so filtering them loses</text>
    <text class="sm" x="690" y="234">nothing in correctness.</text>
    <line x1="650" y1="200" x2="670" y2="200" stroke="#00A972" stroke-width="1.6" marker-end="url(#sk2-arr-g)"></line>
  </svg>
  <div class="note"><strong>Step 2 in the fix order:</strong> after confirming AQE did not resolve the skew, check if null or junk keys are the cause. This is the most common source of artificial skew because every null hashes to the same partition. The fix is a one-line WHERE clause before the join.</div>
</div>

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  o.channel,
  COUNT(*) AS lines,
  SUM(d.quantity) AS units
FROM order_items_dirty d
LEFT JOIN orders o ON d.order_idx = o.order_idx
GROUP BY o.channel
ORDER BY lines DESC
</pre>
</div>

In [ ]:
print("Running SKEW_DIRTY_SQL (open its Query Profile, color by Time) ...")
_dirty_rows = spark.sql(SKEW_DIRTY_SQL).count()
print(f"Returned {_dirty_rows} channel groups. Null order_idx rows piled onto one partition.")

### C3. Filter the Junk Keys Before the Join

The rows with a null **order_idx** can never match, so filtering them costs nothing in correctness and removes the hot partition. Run the fixed query and compare the profile: the straggler is gone, and the null **channel** group (the unmatched junk) disappears from the result.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  o.channel,
  COUNT(*) AS lines,
  SUM(d.quantity) AS units
FROM order_items_dirty d
LEFT JOIN orders o ON d.order_idx = o.order_idx
WHERE d.order_idx IS NOT NULL
GROUP BY o.channel
ORDER BY lines DESC
</pre>
</div>

In [ ]:
print("Running SKEW_FIXED_SQL (compare its profile to C2) ...")
_fixed_rows = spark.sql(SKEW_FIXED_SQL).count()
print(f"Returned {_fixed_rows} channel groups. The null-key partition is gone.")

### C4. Salting

<div style="display:inline-block;background:#618794;color:#ffffff;font-size:11pt;font-weight:700;letter-spacing:1px;text-transform:uppercase;padding:4px 10px;border-radius:4px;margin-bottom:10px;">Last resort, illustration only</div>

When a single legitimate key is hot (not null, not junk, just huge) and AQE has not fully absorbed it, salting spreads that key across buckets. It is a real code change with real costs (extra shuffle, replicated data, harder maintenance), so exhaust AQE and key fixes first. The diagram below shows the concept.

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### How salting works
- Append a random bucket number (the salt) to the hot side's key, so one hot key becomes N warm keys spread across N tasks.
- Replicate the other side across all N buckets so every salted row still finds its match.
- Join on key plus salt, then aggregate. The hot partition is now split N ways.
- The cost is a more complex query and replicated data on the small side, which is why it is the last rung. For most skew, AQE plus a key fix is enough.

### The anti-pattern
- Do not buy a bigger cluster to survive skew. More cores do not help when one task holds most of the rows. Fix the distribution, do not pay for idle capacity waiting on a straggler.

</details>

<div class="skew3">
  <style>
    .skew3 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .skew3 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .skew3 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .skew3 svg { display: block; width: 100%; height: auto; }
    .skew3 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .skew3 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .skew3 .sm { font-size: 12px; fill: #62707d; }
    .skew3 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .skew3 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #98102A; padding: 4px 0 4px 10px; line-height: 1.55; }
    .skew3 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>Salting: Spreading a Hot Key Across Partitions</h3>
  <p class="sub">Append a random suffix so one hot key becomes N warm keys on N different tasks.</p>
  <svg viewBox="0 0 900 260" role="img" aria-label="Diagram showing how salting splits a hot key across multiple partitions">
    <defs>
      <marker id="sk3-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"></path>
      </marker>
    </defs>
    <!-- Left: the hot key -->
    <rect x="0" y="0" width="180" height="250" rx="10" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="tag" x="16" y="22" fill="#98102A">HOT KEY</text>
    <text class="hdr" x="16" y="44">customer_42</text>
    <text class="sm" x="16" y="64">1M rows</text>
    <!-- Stack of rows visualized as bars -->
    <rect x="16" y="78" width="148" height="16" rx="3" fill="#d4808f"></rect>
    <rect x="16" y="98" width="148" height="16" rx="3" fill="#d4808f"></rect>
    <rect x="16" y="118" width="148" height="16" rx="3" fill="#d4808f"></rect>
    <rect x="16" y="138" width="148" height="16" rx="3" fill="#d4808f"></rect>
    <rect x="16" y="158" width="148" height="16" rx="3" fill="#d4808f"></rect>
    <rect x="16" y="178" width="148" height="16" rx="3" fill="#d4808f"></rect>
    <text class="sm" x="16" y="214">All hash to same</text>
    <text class="sm" x="16" y="230">partition (straggler)</text>
    <!-- Arrow 1: add salt -->
    <line x1="180" y1="125" x2="230" y2="125" stroke="#618794" stroke-width="1.8" marker-end="url(#sk3-arr)"></line>
    <text class="tag" x="186" y="110" fill="#618794">+ SALT</text>
    <!-- Middle: salted keys -->
    <rect x="234" y="0" width="220" height="250" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="tag" x="250" y="22" fill="#7a4a2e">SALTED KEY</text>
    <!-- Bucket 0 -->
    <rect x="250" y="36" width="188" height="56" rx="6" fill="#ffffff" stroke="#e5c766"></rect>
    <text class="sm" x="262" y="54">customer_42_0</text>
    <rect x="262" y="60" width="80" height="10" rx="2" fill="#FFAB00"></rect>
    <rect x="346" y="60" width="80" height="10" rx="2" fill="#FFAB00"></rect>
    <text class="sm" x="262" y="86">~333K rows</text>
    <!-- Bucket 1 -->
    <rect x="250" y="98" width="188" height="56" rx="6" fill="#ffffff" stroke="#e5c766"></rect>
    <text class="sm" x="262" y="116">customer_42_1</text>
    <rect x="262" y="122" width="80" height="10" rx="2" fill="#FFAB00"></rect>
    <rect x="346" y="122" width="80" height="10" rx="2" fill="#FFAB00"></rect>
    <text class="sm" x="262" y="148">~333K rows</text>
    <!-- Bucket 2 -->
    <rect x="250" y="160" width="188" height="56" rx="6" fill="#ffffff" stroke="#e5c766"></rect>
    <text class="sm" x="262" y="178">customer_42_2</text>
    <rect x="262" y="184" width="80" height="10" rx="2" fill="#FFAB00"></rect>
    <rect x="346" y="184" width="80" height="10" rx="2" fill="#FFAB00"></rect>
    <text class="sm" x="262" y="210">~333K rows</text>
    <text class="sm" x="250" y="238">rand() assigns each row</text>
    <!-- Arrow 2: shuffle -->
    <line x1="454" y1="125" x2="504" y2="60" stroke="#618794" stroke-width="1.4" marker-end="url(#sk3-arr)"></line>
    <line x1="454" y1="125" x2="504" y2="125" stroke="#618794" stroke-width="1.4" marker-end="url(#sk3-arr)"></line>
    <line x1="454" y1="125" x2="504" y2="190" stroke="#618794" stroke-width="1.4" marker-end="url(#sk3-arr)"></line>
    <!-- Right: three tasks -->
    <rect x="508" y="0" width="390" height="250" rx="10" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="tag" x="524" y="22" fill="#00A972">3 TASKS (BALANCED)</text>
    <!-- Task 0 -->
    <rect x="524" y="36" width="358" height="50" rx="6" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="538" y="56">Task 0: customer_42_0</text>
    <rect x="538" y="64" width="100" height="10" rx="2" fill="#00A972"></rect>
    <!-- Task 1 -->
    <rect x="524" y="98" width="358" height="50" rx="6" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="538" y="118">Task 1: customer_42_1</text>
    <rect x="538" y="126" width="100" height="10" rx="2" fill="#00A972"></rect>
    <!-- Task 2 -->
    <rect x="524" y="160" width="358" height="50" rx="6" fill="#d6ecdd" stroke="#8bbf9c"></rect>
    <text class="sm" x="538" y="180">Task 2: customer_42_2</text>
    <rect x="538" y="188" width="100" height="10" rx="2" fill="#00A972"></rect>
    <!-- Result -->
    <text class="sm" x="524" y="234">Each task handles 333K rows</text>
    <text class="sm" x="524" y="248">instead of one task handling 1M</text>
  </svg>
  <div class="note"><strong>Last resort (Step 3):</strong> The lookup side must be replicated across all N salt values, which increases shuffle. Only use after AQE and key cleanup have failed. The cell below shows the full salting pattern.</div>
</div>

## D. Spill
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">Spill is a task that outgrew its memory</div>
    <div class="ds-banner-tagline">When a task's working set will not fit in memory, it writes the overflow to local disk.</div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The de-teach: adding workers does not fix spill
- Scaling out adds cores but not per-core memory. If one task is too big for its slot, more machines just means more idle cores waiting on the straggler.
- The old habit of hand-setting `spark.sql.shuffle.partitions = 200` made spill worse by creating fewer, larger partitions. That default is retired; `auto` is the standard.

</details>

<div class="spill1">
  <style>
    .spill1 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .spill1 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .spill1 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .spill1 svg { display: block; width: 100%; height: auto; }
    .spill1 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .spill1 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .spill1 .sm { font-size: 12px; fill: #62707d; }
    .spill1 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .spill1 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #4299E0; padding: 4px 0 4px 10px; line-height: 1.55; }
    .spill1 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>What Spill Looks Like</h3>
  <p class="sub">A task receives more data than fits in its memory allocation. The excess is written to local disk.</p>
  <svg viewBox="0 0 900 290" role="img" aria-label="Diagram showing how spill occurs when a task data exceeds available memory">
    <defs>
      <marker id="sp1-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"></path>
      </marker>
      <marker id="sp1-arr-r" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#98102A"></path>
      </marker>
    </defs>
    <!-- Left: one partition arriving -->
    <rect x="0" y="0" width="200" height="280" rx="10" fill="#F8F9FC" stroke="#94bee0"></rect>
    <text class="hdr" x="16" y="24">ONE PARTITION</text>
    <text class="sm" x="16" y="44">arriving at a single task</text>
    <!-- Single block: one partition, one chunk of data -->
    <rect x="16" y="60" width="168" height="200" rx="6" fill="#4299E0" fill-opacity="0.2" stroke="#4299E0" stroke-width="1.5"></rect>
    <text class="sm" x="100" y="165" text-anchor="middle" style="font-size:18px; fill:#1B5162; font-weight:700;">6 GB</text>
    <!-- Arrow -->
    <line x1="200" y1="140" x2="248" y2="140" stroke="#618794" stroke-width="1.8" marker-end="url(#sp1-arr)"></line>
    <!-- Middle: task memory container -->
    <rect x="252" y="0" width="230" height="280" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="hdr" x="268" y="24">TASK MEMORY (4 GB)</text>
    <!-- Memory box -->
    <rect x="272" y="44" width="190" height="160" rx="6" fill="#ffffff" stroke="#d3dde5" stroke-width="1.5"></rect>
    <!-- Blue fill (what fits) -->
    <rect x="273" y="45" width="188" height="158" rx="5" fill="#4299E0" fill-opacity="0.15"></rect>
    <text class="sm" x="367" y="130" text-anchor="middle" style="font-size:14px; fill:#4299E0; font-weight:600;">4 GB fits here</text>
    <!-- Overflow badge -->
    <rect x="272" y="216" width="190" height="50" rx="6" fill="#98102A" fill-opacity="0.12" stroke="#98102A" stroke-dasharray="4 2"></rect>
    <text class="tag" x="367" y="238" fill="#98102A" text-anchor="middle">2 GB OVERFLOW</text>
    <text class="sm" x="367" y="254" text-anchor="middle">written to local disk</text>
    <!-- Arrow down -->
    <line x1="367" y1="266" x2="367" y2="282" stroke="#98102A" stroke-width="1.8" marker-end="url(#sp1-arr-r)"></line>
    <!-- Right: what the profile shows -->
    <rect x="506" y="0" width="392" height="280" rx="10" fill="#F8F9FC" stroke="#94bee0"></rect>
    <text class="hdr" x="522" y="24">WHAT THE PROFILE SHOWS</text>
    <!-- Metric pair -->
    <rect x="522" y="40" width="360" height="60" rx="6" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="536" y="58" fill="#4299E0">METRIC PAIR</text>
    <text class="sm" x="536" y="76">Spill (memory): in-memory size of overflow</text>
    <text class="sm" x="536" y="92">Spill (disk): compressed bytes written to disk</text>
    <!-- system_vw.query.history -->
    <rect x="522" y="110" width="360" height="46" rx="6" fill="#ffffff" stroke="#d3dde5"></rect>
    <text class="tag" x="536" y="128" fill="#4299E0">IN QUERY HISTORY</text>
    <text class="sm" x="536" y="146">spilled_local_bytes &gt; 0</text>
    <!-- Insight badge -->
    <rect x="522" y="168" width="360" height="96" rx="8" fill="#4299E0" fill-opacity="0.1" stroke="#4299E0"></rect>
    <text class="tag" x="538" y="190" fill="#1B5162">PERFORMANCE INSIGHT: DATA_SPILL</text>
    <text class="sm" x="538" y="212">"Data was spilled to disk during</text>
    <text class="sm" x="538" y="228">execution, which degrades</text>
    <text class="sm" x="538" y="244">performance."</text>
  </svg>
  <div class="note"><strong>Why disk is slow:</strong> Memory throughput is measured in GB/s. Local disk is 10 to 100 times slower. Every byte that spills must be serialized, written, then read back and deserialized. That round-trip shows up as added wall-clock time on the task.</div>
</div>

<div class="spill2">
  <style>
    .spill2 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .spill2 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .spill2 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .spill2 svg { display: block; width: 100%; height: auto; }
    .spill2 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .spill2 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .spill2 .sm { font-size: 12px; fill: #62707d; }
    .spill2 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .spill2 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #00A972; padding: 4px 0 4px 10px; line-height: 1.55; }
    .spill2 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>What Fixes Spill</h3>
  <p class="sub">Reduce the data per task, or increase the memory per task. The platform handles the first one automatically.</p>
  <svg viewBox="0 0 900 300" role="img" aria-label="Diagram showing three approaches to fixing spill">
    <!-- Column 1: Auto partitions (green) -->
    <rect x="0" y="0" width="280" height="290" rx="10" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="tag" x="16" y="22" fill="#00A972">FIX 1: SMALLER PARTITIONS</text>
    <text class="hdr" x="16" y="44">shuffle.partitions = auto</text>
    <text class="sm" x="16" y="62">(already the default)</text>
    <!-- Before: one big partition -->
    <text class="tag" x="16" y="88" fill="#62707d">BEFORE</text>
    <rect x="16" y="96" width="248" height="28" rx="4" fill="#FFAB00" fill-opacity="0.3" stroke="#FFAB00"></rect>
    <text class="sm" x="80" y="114">1 large partition (spills)</text>
    <!-- Arrow -->
    <text class="sm" x="130" y="142">↓</text>
    <!-- After: many small partitions -->
    <text class="tag" x="16" y="160" fill="#62707d">AFTER</text>
    <rect x="16" y="168" width="58" height="28" rx="4" fill="#00A972" fill-opacity="0.2" stroke="#00A972"></rect>
    <rect x="80" y="168" width="58" height="28" rx="4" fill="#00A972" fill-opacity="0.2" stroke="#00A972"></rect>
    <rect x="144" y="168" width="58" height="28" rx="4" fill="#00A972" fill-opacity="0.2" stroke="#00A972"></rect>
    <rect x="208" y="168" width="56" height="28" rx="4" fill="#00A972" fill-opacity="0.2" stroke="#00A972"></rect>
    <text class="sm" x="40" y="214">4 smaller partitions (each fits)</text>
    <!-- Explanation -->
    <text class="sm" x="16" y="244">AQE splits oversized partitions</text>
    <text class="sm" x="16" y="260">at runtime. More tasks, less</text>
    <text class="sm" x="16" y="276">data per task, no spill.</text>
    <!-- Column 2: More memory per core (blue) -->
    <rect x="296" y="0" width="280" height="290" rx="10" fill="#F8F9FC" stroke="#94bee0"></rect>
    <text class="tag" x="312" y="22" fill="#1B5162">FIX 2: MORE MEMORY PER CORE</text>
    <text class="hdr" x="312" y="44">Bigger instance type</text>
    <text class="sm" x="312" y="62">(classic compute only)</text>
    <!-- Before: small memory box, data overflows -->
    <text class="tag" x="312" y="88" fill="#62707d">BEFORE (4 GB per core)</text>
    <rect x="312" y="96" width="110" height="70" rx="4" fill="#ffffff" stroke="#d3dde5"></rect>
    <rect x="312" y="136" width="110" height="30" rx="0" fill="#4299E0" fill-opacity="0.2"></rect>
    <line x1="312" y1="136" x2="422" y2="136" stroke="#98102A" stroke-width="1.5" stroke-dasharray="4 2"></line>
    <rect x="312" y="96" width="110" height="40" rx="0" fill="#98102A" fill-opacity="0.1"></rect>
    <text class="tag" x="432" y="120" fill="#98102A">SPILLS</text>
    <!-- After: larger memory box -->
    <text class="tag" x="312" y="186" fill="#62707d">AFTER (16 GB per core)</text>
    <rect x="312" y="194" width="180" height="70" rx="4" fill="#ffffff" stroke="#94bee0"></rect>
    <rect x="312" y="234" width="180" height="30" rx="0" fill="#4299E0" fill-opacity="0.2"></rect>
    <line x1="312" y1="214" x2="492" y2="214" stroke="#00A972" stroke-width="1.5" stroke-dasharray="4 2"></line>
    <text class="tag" x="502" y="218" fill="#00A972">FITS</text>
    <text class="sm" x="312" y="280">Same data, larger container.</text>
    <!-- Column 3: Fix root cause (amber) -->
    <rect x="592" y="0" width="308" height="290" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="tag" x="608" y="22" fill="#7a4a2e">FIX 3: FIX THE ROOT CAUSE</text>
    <text class="hdr" x="608" y="44">Spill is often a symptom</text>
    <text class="sm" x="608" y="62">(persistent spill = look deeper)</text>
    <!-- Skew -->
    <rect x="608" y="80" width="276" height="86" rx="6" fill="#ffffff" stroke="#e5c766"></rect>
    <text class="tag" x="622" y="100" fill="#FFAB00">SKEW (Section C)</text>
    <text class="sm" x="622" y="118">One task holds most of the data.</text>
    <text class="sm" x="622" y="134">Fix the key distribution and</text>
    <text class="sm" x="622" y="150">the spill disappears.</text>
    <!-- Explosion -->
    <rect x="608" y="178" width="276" height="86" rx="6" fill="#ffffff" stroke="#e5c766"></rect>
    <text class="tag" x="622" y="198" fill="#FFAB00">EXPLOSION (Section E)</text>
    <text class="sm" x="622" y="216">A fan-out join multiplies rows.</text>
    <text class="sm" x="622" y="232">Reduce the multiplication and</text>
    <text class="sm" x="622" y="248">data volume drops.</text>
    <text class="sm" x="608" y="280">Treat the cause, not the symptom.</text>
  </svg>
  <div class="note"><strong>Key point:</strong> Adding more workers does not fix spill. More cores spread tasks across machines, but each core still gets the same memory allocation. If one task is too big for its slot, you need fewer bytes per task (Fix 1, Fix 3) or more memory per task (Fix 2).</div>
</div>

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Watch the precursor, not the spill</div>
  <div class="ds-callout-title">On serverless at demo scale, you won't see real spill</div>
  <ul>
    <li>Photon on serverless allocates per-task memory proportional to data volume, so at demo scale <code>spilled_local_bytes</code> stays at <strong>0</strong>.</li>
    <li>What you observe instead is the <strong>peak shuffle memory</strong> in the profile's memory-peak view. That high-water mark is the leading indicator, the precursor that becomes spill when memory is fixed.</li>
    <li>On <strong>classic compute</strong> with fixed executor memory, or at production volumes, that same pressure crosses the line into <code>spilled_local_bytes &gt; 0</code>.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why serverless doesn't spill here
- Serverless allocates memory per task in proportion to the data each task handles. As data grows, so does the allocation, so spill does not appear at demo scale (verified up to hundreds of millions of rows).
- The real signal is the **peak shuffle memory** high-water mark. Color the profile by **Memory peak** and read the largest shuffle. That is the precursor: it's how close a task came to running out of memory.

</details>

### D2. Rank Recent Queries by Shuffle and Spill

The helper below reads `system_vw.query.history` for your recent queries and shows their shuffle and spill metrics. Expect `spilled_local_bytes = 0` on serverless. 

The queries from sections B and C moved real shuffle bytes without spilling, because the platform absorbed the memory pressure.

There's a 5 to 10 minute ingestion lag. If the queries you just ran are missing, wait a moment and re-run this cell.

In [ ]:
spill_shuffle_history()

## E. Explosion
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.caution { --bnr-bg: rgba(255,171,0,0.12); --bnr-accent: #FFAB00; --bnr-icon-fg: #0b2026; --bnr-title: #0b2026; --bnr-tagline: #5E7077; }
</style>

<div class="ds-banner caution">
  <div class="ds-banner-icon">!</div>
  <div>
    <div class="ds-banner-title">Explosion is rows out far exceeding rows in</div>
    <div class="ds-banner-tagline">A fan-out join multiplies rows mid-query. A modest input becomes something much larger that then has to shuffle.</div>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Two causes, two fixes
- **Legitimate fan-out** (one-to-many relationship): orders to items, users to events. Fix by pre-aggregating on the many-side before the join so fewer rows enter.
- **Duplicate-key join** (data-quality bug): repeated keys multiply rows unintentionally. Fix by deduplicating before the join.

</details>

<div class="expl1">
  <style>
    .expl1 { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1060px; }
    .expl1 h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .expl1 p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .expl1 svg { display: block; width: 100%; height: auto; }
    .expl1 text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .expl1 .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .expl1 .sm { font-size: 12px; fill: #62707d; }
    .expl1 .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .expl1 .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #FFAB00; padding: 4px 0 4px 10px; line-height: 1.55; }
    .expl1 code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>How Explosion Works</h3>
  <p class="sub">A one-to-many join multiplies rows. The output is much larger than the input, and all those extra rows must then shuffle.</p>
  <svg viewBox="0 0 900 300" role="img" aria-label="Diagram showing how a fan-out join multiplies rows mid-query">
    <defs>
      <marker id="ex1-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"></path>
      </marker>
    </defs>
    <!-- Left: input (orders) -->
    <rect x="0" y="0" width="180" height="290" rx="10" fill="#F8F9FC" stroke="#94bee0"></rect>
    <text class="hdr" x="16" y="24">INPUT</text>
    <text class="tag" x="16" y="44" fill="#4299E0">orders</text>
    <!-- Single order block -->
    <rect x="16" y="58" width="148" height="180" rx="6" fill="#4299E0" fill-opacity="0.15" stroke="#4299E0"></rect>
    <text class="sm" x="90" y="120" text-anchor="middle" style="font-size:28px; fill:#1B5162; font-weight:700;">10M</text>
    <text class="sm" x="90" y="144" text-anchor="middle">rows</text>
    <text class="sm" x="90" y="170" text-anchor="middle">(one row per order)</text>
    <text class="sm" x="90" y="258" text-anchor="middle">Compact. Manageable.</text>
    <!-- Arrow -->
    <line x1="180" y1="145" x2="220" y2="145" stroke="#618794" stroke-width="1.8" marker-end="url(#ex1-arr)"></line>
    <!-- Middle: the join (fan-out) -->
    <rect x="224" y="0" width="220" height="290" rx="10" fill="#fffbf0" stroke="#e5c766"></rect>
    <text class="hdr" x="240" y="24">JOIN</text>
    <text class="tag" x="240" y="44" fill="#FFAB00">order_items (1-to-many)</text>
    <!-- One order becoming 6 items -->
    <rect x="240" y="62" width="80" height="40" rx="4" fill="#4299E0" fill-opacity="0.2" stroke="#4299E0"></rect>
    <text class="sm" x="280" y="86" text-anchor="middle">1 order</text>
    <!-- Fan-out arrows -->
    <line x1="320" y1="72" x2="350" y2="64" stroke="#FFAB00" stroke-width="1.2"></line>
    <line x1="320" y1="77" x2="350" y2="77" stroke="#FFAB00" stroke-width="1.2"></line>
    <line x1="320" y1="82" x2="350" y2="90" stroke="#FFAB00" stroke-width="1.2"></line>
    <line x1="320" y1="87" x2="350" y2="103" stroke="#FFAB00" stroke-width="1.2"></line>
    <line x1="320" y1="92" x2="350" y2="116" stroke="#FFAB00" stroke-width="1.2"></line>
    <line x1="320" y1="97" x2="350" y2="129" stroke="#FFAB00" stroke-width="1.2"></line>
    <!-- 6 item rows -->
    <rect x="350" y="58" width="80" height="16" rx="3" fill="#FFAB00" fill-opacity="0.3" stroke="#FFAB00"></rect>
    <rect x="350" y="76" width="80" height="16" rx="3" fill="#FFAB00" fill-opacity="0.3" stroke="#FFAB00"></rect>
    <rect x="350" y="94" width="80" height="16" rx="3" fill="#FFAB00" fill-opacity="0.3" stroke="#FFAB00"></rect>
    <rect x="350" y="112" width="80" height="16" rx="3" fill="#FFAB00" fill-opacity="0.3" stroke="#FFAB00"></rect>
    <rect x="350" y="130" width="80" height="16" rx="3" fill="#FFAB00" fill-opacity="0.3" stroke="#FFAB00"></rect>
    <text class="sm" x="390" y="162" text-anchor="middle">6 item rows</text>
    <!-- Multiplier badge -->
    <rect x="256" y="182" width="172" height="90" rx="6" fill="#FFAB00" fill-opacity="0.12" stroke="#FFAB00"></rect>
    <text class="tag" x="342" y="204" text-anchor="middle" fill="#7a4a2e">MULTIPLICATION</text>
    <text class="sm" x="342" y="224" text-anchor="middle">Each order has ~6 items</text>
    <text class="sm" x="342" y="240" text-anchor="middle">10M x 6 = 60M rows</text>
    <text class="sm" x="342" y="256" text-anchor="middle">flowing through the join</text>
    <!-- Arrow to output -->
    <line x1="444" y1="145" x2="484" y2="145" stroke="#618794" stroke-width="1.8" marker-end="url(#ex1-arr)"></line>
    <!-- Right: output -->
    <rect x="488" y="0" width="410" height="290" rx="10" fill="#fdf2f4" stroke="#d4808f"></rect>
    <text class="hdr" x="504" y="24">OUTPUT (after join, before aggregate)</text>
    <!-- Large block representing 60M -->
    <rect x="504" y="42" width="180" height="230" rx="6" fill="#FF5F46" fill-opacity="0.15" stroke="#FF5F46"></rect>
    <text class="sm" x="594" y="130" text-anchor="middle" style="font-size:28px; fill:#98102A; font-weight:700;">60M</text>
    <text class="sm" x="594" y="154" text-anchor="middle">rows</text>
    <text class="sm" x="594" y="180" text-anchor="middle">(all must shuffle)</text>
    <!-- Consequence -->
    <rect x="700" y="42" width="186" height="230" rx="6" fill="#ffffff" stroke="#d4808f"></rect>
    <text class="tag" x="716" y="64" fill="#98102A">CONSEQUENCES</text>
    <text class="sm" x="716" y="88">6x more shuffle bytes</text>
    <text class="sm" x="716" y="108">6x more memory pressure</text>
    <text class="sm" x="716" y="128">6x longer aggregation</text>
    <text class="sm" x="716" y="160">The aggregate collapses</text>
    <text class="sm" x="716" y="176">rows back down, but the</text>
    <text class="sm" x="716" y="192">damage is already done:</text>
    <text class="sm" x="716" y="208">all 60M rows already</text>
    <text class="sm" x="716" y="224">shuffled.</text>
    <!-- Profile hint -->
    <rect x="504" y="278" width="380" height="0" rx="0" fill="none"></rect>
  </svg>
  <div class="note"><strong>How to spot it:</strong> Color the Query Profile by <strong>Rows</strong>. The count jumps at the join operator (10M in, 60M out), then drops back at the aggregate. The <strong>EXPLODING_JOIN</strong> insight fires automatically: "The join produces significantly more rows than it reads."</div>
</div>

### E1. Join Then Aggregate vs Pre-Aggregate Then Join

Both queries answer the same question (total units per customer).

This query joins orders to order_items first, then aggregates. Every order fans out to its ~6 line items, so 10M orders become ~60M rows flowing through the join before the GROUP BY collapses them.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  o.customer_idx,
  SUM(oi.quantity) AS total_units
FROM orders o
JOIN order_items oi ON o.order_idx = oi.order_idx
GROUP BY o.customer_idx
</pre>
</div>

In [ ]:
print("Running EXPLOSION_JOIN_SQL (color the profile by Rows) ...")
_expl_stats = (spark.sql(EXPLOSION_JOIN_SQL)
               .agg(F.count("*").alias("customers"),
                    F.sum("total_units").alias("grand_total"))
               .first())
print(f"Returned {_expl_stats['customers']:,} customers, {_expl_stats['grand_total']:,} total units.")
print("~60M item rows flowed through the join first.")

The fix is to re-aggregate order_items to one row per order *before* the join. Now only 10M rows cross the join instead of 60M.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
WITH per_order AS (
  SELECT order_idx, SUM(quantity) AS order_units
  FROM order_items
  GROUP BY order_idx
)
SELECT
  o.customer_idx,
  SUM(po.order_units) AS total_units
FROM orders o
JOIN per_order po ON o.order_idx = po.order_idx
GROUP BY o.customer_idx
</pre>
</div>

In [ ]:
print("Running EXPLOSION_PREAGG_SQL (compare Rows and shuffle_read_bytes) ...")
_preagg_stats = (spark.sql(EXPLOSION_PREAGG_SQL)
                 .agg(F.count("*").alias("customers"),
                      F.sum("total_units").alias("grand_total"))
                 .first())
print(f"Returned {_preagg_stats['customers']:,} customers, {_preagg_stats['grand_total']:,} total units.")
print("Pre-aggregation sends ~10M rows across the join, not ~60M.")

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <div style="border-left: 4px solid #4299E0; background: #F8F9FC; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
    <strong style="display:block; color:#1B5162; margin-bottom:6px; font-size:15pt;">Compare the Two Query Profiles</strong>
    <div style="color:#0b2026; font-size:14pt; line-height:1.55;">
      Click the statement link in each cell's output to open its Query Profile. Set <strong>Color by → Rows</strong> on both, then compare:
      <ol style="margin-top:8px; padding-left:20px;">
        <li><strong>Inner Join operator</strong>: the join-first profile shows ~60M output rows at the join node. The pre-agg profile shows ~10M, because each order is already one row.</li>
        <li><strong>Grouping Aggregate</strong>: in the join-first plan, the aggregate processes all ~60M post-join rows. In the pre-agg plan, it processes only ~10M.</li>
      </ol>
    </div>
  </div>
</div>

### E2. Higher-Order Functions

When you only need a per-row metric over an array, you do not need to explode at all. 

Higher-order functions (`transform`, `filter`, `aggregate`, `size`) compute directly on the array, avoiding the explode-then-`collect_list` round trip.

Instead of exploding the arrays and re-aggregating (which multiplies rows), the query uses higher-order functions to compute directly on each array in place.

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">
SELECT
  order_idx,
  size(product_idxs) AS item_count,
  aggregate(quantities, 0, (acc, x) -> acc + x) AS total_units,
  size(filter(quantities, q -> q >= 3)) AS bulk_lines
FROM order_baskets
ORDER BY total_units DESC
LIMIT 20
</pre>
</div>

In [ ]:
# Explode: fan out arrays to rows, GROUP BY to collapse
print("Explode approach ...")
_explode_stats = (spark.sql(f"""
SELECT order_idx,
       COUNT(*) AS item_count,
       SUM(q) AS total_units,
       COUNT_IF(q >= 3) AS bulk_lines
FROM {_S}.order_baskets,
     LATERAL explode(quantities) AS t(q)
GROUP BY order_idx
""").agg(F.count("*").alias("orders"),
         F.sum("total_units").alias("grand_total"))
   .first())
print(f"  {_explode_stats['orders']:,} orders, {_explode_stats['grand_total']:,} units (~6M intermediate rows).")

# HOF: process arrays in place
print("\nHOF approach ...")
_hof_stats = (spark.sql(f"""
SELECT order_idx,
       size(product_idxs) AS item_count,
       aggregate(quantities, 0, (acc, x) -> acc + x) AS total_units,
       size(filter(quantities, q -> q >= 3)) AS bulk_lines
FROM {_S}.order_baskets
""").agg(F.count("*").alias("orders"),
         F.sum("total_units").alias("grand_total"))
   .first())
print(f"  {_hof_stats['orders']:,} orders, {_hof_stats['grand_total']:,} units (zero row expansion).")

if _explode_stats['grand_total'] == _hof_stats['grand_total']:
    print(f"\n\u2713 Same answer. Open both profiles and color by Rows to see the difference.")
else:
    print(f"\n\u2717 Totals differ! Explode: {_explode_stats['grand_total']:,}  HOF: {_hof_stats['grand_total']:,}")

<div style="width: 100%; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:64px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/><path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/><path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/><path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/><rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/><path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/><path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/><path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">You have already used <strong>Genie Code</strong> to optimize a query from the Query Profile in Part 2. Now that you can name the four patterns yourself, use it as an accelerator: paste a Performance Insight and ask for the cheapest fix, then judge the answer against what you just learned. Click <strong>Copy</strong> and paste the prompt into Genie Code.</div>
  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #EEEDE9; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">Explain this Performance Insight on my query and suggest the cheapest remediation. Confirm whether AQE, auto shuffle partitions, or a broadcast join already handled part of it before recommending a code change.</span>
    <button onclick="var text=document.getElementById('genie-query').innerText; var ta=document.createElement('textarea'); ta.value=text; ta.style.position='fixed'; ta.style.opacity='0'; document.body.appendChild(ta); ta.select(); document.execCommand('copy'); document.body.removeChild(ta); var b=this; b.innerText='Copied!'; setTimeout(function(){ b.innerText='Copy'; }, 2000);" style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 6px 12px; font-size: 14pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>
</div>
</div>

## Conclusion

You profiled each of the four slow-down patterns and applied the cheapest available fix:

- **Shuffle**: compared a shuffle hash join (779 MB network transfer, 4.5s) against a broadcast join (0 bytes, 1.65s) on the same data. The difference was table size: dimensions above 10 MB shuffle, the 4-row lookup broadcasts automatically.
- **Skew**: saw 80% of orders concentrate on 1% of customers. Filtered null keys from `order_items_dirty` to remove the straggler task (a one-line WHERE clause). Salting was shown as the last resort when AQE and key cleanup fail.
- **Spill**: identified peak shuffle memory as the serverless precursor to real spill. The fix order is smaller partitions (automatic via AQE), more memory per core (classic compute), or fixing the upstream skew/explosion that caused the oversized partition.
- **Explosion**: the orders-to-items join fanned 10M rows into 60M. Pre-aggregating order_items to one row per order before the join eliminated the multiplication. Higher-order functions (`aggregate`, `filter`, `size`) processed array columns without any row expansion at all.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>